# DA-VA Analysis
## 1 Load model level data


In [1]:
from scipy import stats
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import re
import pandasql as ps
from scipy.stats import wilcoxon
import pandas as pd

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Specify the experiment name and date to analyze
# Set experiment_name to None to analyze all experiments
experiment_name = "monitoring_vs_consent_based_analysis"
experiment_date = "20251119"  # Format: YYYYMMDD
# Note: The actual data is in the directory: monitoring_vs_consent_based_analysis_20251108


In [2]:
def extract_experiment_info(config_filename):
    """Extract experiment name and configuration from config filename.
    
    Example: consent_or_goal_sensitivity_analysis_(seed_2)_seed_2:_0-1000-0_20251013_165148_config.json
    Returns: ('consent_or_goal_sensitivity_analysis', '0-1000-0', '2')
    """
    # Remove _config.json suffix
    name = config_filename.replace('_config.json', '')
    
    # Pattern: {experiment_name}_(seed_{N})_seed_{N}:_{agent_config}_{timestamp}
    # Match the experiment name (everything before _(seed_)
    match = re.match(r'(.+?)_\(seed_(\d+)\)_seed_\2:_(.+?)_(\d{8}_\d{6})$', name)
    
    if match:
        exp_name = match.group(1)
        seed = match.group(2)
        agent_config = match.group(3)
        timestamp_date = match.group(4).split('_')[0]
        return exp_name, agent_config, seed, timestamp_date
    
    return None, None, None

def create_figures_directory(experiment_name, experiment_date):
    """Create figures directory for the experiment if it doesn't exist."""
    results_dir = Path("/Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results")
    
    # Find the experiment directory (it might have a different name than expected)
    experiment_dir = None
    for subdir in results_dir.iterdir():
        if subdir.is_dir():
            # Check if this directory contains files matching our experiment name and date
            configs_dir = subdir / "configs"
            if configs_dir.exists():
                for config_file in configs_dir.glob("*.json"):
                    exp_name, agent_config, seed, file_date = extract_experiment_info(config_file.name)
                    if exp_name == experiment_name and file_date == experiment_date:
                        experiment_dir = subdir
                        break
                if experiment_dir:
                    break
    
    if experiment_dir:
        figures_dir = experiment_dir / "figures"
        figures_dir.mkdir(exist_ok=True)
        return figures_dir
    else:
        # Fallback: create in main results directory
        figures_dir = results_dir / "figures"
        figures_dir.mkdir(exist_ok=True)
        return figures_dir

def load_simulation_data(experiment_name=None, experiment_date=None):
    """Load all simulation data and extract agent ratios."""
    results_dir = Path("/Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results")
    
    # Find all config files - check both main directories and experiment-specific subdirectories
    config_files = []
    
    # Check main configs directory
    main_configs_dir = results_dir / "configs"
    if main_configs_dir.exists():
        config_files.extend(list(main_configs_dir.glob("*.json")))
    
    # Check experiment-specific subdirectories
    if experiment_name and experiment_date:
        exp_subdir = results_dir / f"{experiment_name}_{experiment_date}"
        if exp_subdir.exists():
            exp_configs_dir = exp_subdir / "configs"
            if exp_configs_dir.exists():
                config_files.extend(list(exp_configs_dir.glob("*.json")))
                print(f"Found experiment-specific configs in: {exp_configs_dir}")
    
    # Also search all subdirectories for files that match the experiment name and date
    if experiment_name and experiment_date:
        for subdir in results_dir.iterdir():
            if subdir.is_dir():
                exp_configs_dir = subdir / "configs"
                if exp_configs_dir.exists():
                    # Check if any files in this directory match our criteria
                    matching_files = []
                    for config_file in exp_configs_dir.glob("*.json"):
                        exp_name, agent_config, seed, file_date = extract_experiment_info(config_file.name)
                        if exp_name == experiment_name and file_date == experiment_date:
                            matching_files.append(config_file)
                    
                    if matching_files:
                        config_files.extend(matching_files)
                        print(f"Found matching configs in subdirectory: {exp_configs_dir} ({len(matching_files)} files)")
    
    simulation_data = []
    timestamp_date = None
    
    for config_file in config_files:
        # Extract experiment info from filename
        exp_name, agent_config, seed, timestamp_date = extract_experiment_info(config_file.name)
        
        # Skip if we can't parse the filename or if it doesn't match the desired experiment
        if exp_name is None:
            print(f"Warning: Could not parse filename: {config_file.name}")
            continue
        
        if experiment_name is not None and exp_name != experiment_name:
            continue
        
        # Filter by experiment date if specified
        if experiment_date is not None and timestamp_date != experiment_date:
            continue
        
        # Load config
        with open(config_file, 'r') as f:
            config = json.load(f)
        
        # Extract agent counts
        params = config['parameters']
        consent_first = params.get('ConsentFirstAgent_COUNT', 0)
        monitoring = params.get('MonitoringAgent_COUNT', 0)
        fifty_fifty = params.get('FiftyFiftyAgent_COUNT', 0)
        total_agents = consent_first + monitoring + fifty_fifty
        
        # Calculate ratios
        consent_ratio = consent_first / total_agents if total_agents > 0 else 0
        monitoring_ratio = monitoring / total_agents if total_agents > 0 else 0
        fifty_fifty_ratio = fifty_fifty / total_agents if total_agents > 0 else 0
        
        # Find corresponding model data file
        config_name = config_file.stem
        prefix = config_name.rsplit('_', 1)[0]
        
        # Look for data files in multiple locations
        model_file = None
        agent_file = None
        
        # List of directories to check for data files
        data_dirs_to_check = []
        
        # Check main data directory first
        main_data_dir = results_dir / "data"
        if main_data_dir.exists():
            data_dirs_to_check.append(main_data_dir)
        
        # Check experiment-specific subdirectory
        if experiment_name and experiment_date:
            exp_subdir = results_dir / f"{experiment_name}_{experiment_date}"
            if exp_subdir.exists():
                exp_data_dir = exp_subdir / "data"
                if exp_data_dir.exists():
                    data_dirs_to_check.append(exp_data_dir)
        
        # Also search all subdirectories for data files that match the experiment name and date
        if experiment_name and experiment_date:
            for subdir in results_dir.iterdir():
                if subdir.is_dir():
                    exp_data_dir = subdir / "data"
                    if exp_data_dir.exists() and exp_data_dir not in data_dirs_to_check:
                        # Check if any files in this directory match our criteria
                        # We'll check by looking for files with the same prefix as our config file
                        data_dirs_to_check.append(exp_data_dir)
        
        # Find the first directory that contains the required files
        for data_dir in data_dirs_to_check:
            model_file = data_dir / f"{prefix}_model.csv"
            agent_file = data_dir / f"{prefix}_agents.csv"
            if model_file.exists() and agent_file.exists():
                break
        
        if model_file and model_file.exists():
            # Load model data
            model_df = pd.read_csv(model_file)
            agent_df = pd.read_csv(agent_file)

            # Calculate CI state ratios
            # Handle division by zero
            model_df["Consent Violation Ratio"] = model_df["Total Violated Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Consent Fulfillment Ratio"] = model_df["Total Fulfilled Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Consent Unrealized Ratio"] = model_df["Total Unrealized Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Consent Deferred Ratio"] = model_df["Total Deferred Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Resource Conflict Counter Goal Accomplishment Ratio"] = model_df["Total Resource Conflicts"] / model_df["Total Resource Conflict Accomplished Counter Goals"].replace(0, np.nan)
            
            # Exclude the last early_stop_steps - 1 steps before getting final values
            # But here we should also check if no additional goals were really accomplished after the early stop steps.
            early_stop_steps = config.get('early_stop_steps', 0)
            if early_stop_steps > 1:
                # Exclude the last (early_stop_steps - 1) rows
                steps_to_exclude = early_stop_steps - 1
                distinct_agent_count_q = """SELECT COUNT(DISTINCT AgentID) as AGENT_COUNT FROM agent_df WHERE Step = 1"""
                distinct_agent_count = ps.sqldf(distinct_agent_count_q, locals())["AGENT_COUNT"][0]
                if len(model_df) > steps_to_exclude and model_df.iloc[-steps_to_exclude]['Total Accomplished Goals'] == model_df.iloc[-1]['Total Accomplished Goals']:
                    model_df = model_df.iloc[:-steps_to_exclude]
                    agent_df = agent_df.iloc[:-steps_to_exclude*distinct_agent_count]
                    
            
            # Get final values (last distinct_agent_count rows after exclusion)
            final_agent_values = agent_df.iloc[-distinct_agent_count:]
            
            # Calculate agent-level metrics for each agent type
            consent_first_mask = final_agent_values['Agent Persona'] == 'ConsentFirstAgent'
            monitoring_mask = final_agent_values['Agent Persona'] == 'MonitoringAgent'
            
            # Calculate consent-related metrics from the available columns
            # Note: The CSV has different column names than expected
            avg_accomplished_goals_consent_first_agent = final_agent_values[consent_first_mask]['Accomplished Goals'].mean() if consent_first_mask.any() else 0
            avg_accomplished_goals_monitoring_agent = final_agent_values[monitoring_mask]['Accomplished Goals'].mean() if monitoring_mask.any() else 0
            avg_remaining_goals_consent_first_agent = final_agent_values[consent_first_mask]['Remaining Goals'].mean() if consent_first_mask.any() else 0
            avg_remaining_goals_monitoring_agent = final_agent_values[monitoring_mask]['Remaining Goals'].mean() if monitoring_mask.any() else 0
            avg_resource_conflicts_consent_first_agent = final_agent_values[consent_first_mask]['Resource Conflicts'].mean() if consent_first_mask.any() else 0
            avg_resource_conflicts_monitoring_agent = final_agent_values[monitoring_mask]['Resource Conflicts'].mean() if monitoring_mask.any() else 0
            avg_counter_goal_accomplishments_consent_first_agent = final_agent_values[consent_first_mask]['Counter Conflict Goal Accomplishments'].mean() if consent_first_mask.any() else 0
            avg_counter_goal_accomplishments_monitoring_agent = final_agent_values[monitoring_mask]['Counter Conflict Goal Accomplishments'].mean() if monitoring_mask.any() else 0
            
            # Calculate consent metrics from available columns
            # Separately for R (Receiver) and G (Giver) and agent type.
            total_consents_consent_first_r = final_agent_values[consent_first_mask]['Number of Consents as R'].mean() if consent_first_mask.any() else 0
            total_consents_consent_first_g = final_agent_values[consent_first_mask]['Number of Consents as G'].mean() if consent_first_mask.any() else 0
            total_consents_monitoring_r = final_agent_values[monitoring_mask]['Number of Consents as R'].mean() if monitoring_mask.any() else 0
            total_consents_monitoring_g = final_agent_values[monitoring_mask]['Number of Consents as G'].mean() if monitoring_mask.any() else 0
            violated_consents_consent_first_r = final_agent_values[consent_first_mask]['Number of Consents as R Violated'].mean() if consent_first_mask.any() else 0
            violated_consents_consent_first_g = final_agent_values[consent_first_mask]['Number of Consents as G Violated'].mean() if consent_first_mask.any() else 0
            violated_consents_monitoring_r = final_agent_values[monitoring_mask]['Number of Consents as R Violated'].mean() if monitoring_mask.any() else 0
            violated_consents_monitoring_g = final_agent_values[monitoring_mask]['Number of Consents as G Violated'].mean() if monitoring_mask.any() else 0
            
            fulfilled_consents_consent_first_r = final_agent_values[consent_first_mask]['Number of Consents as R Fulfilled'].mean() if consent_first_mask.any() else 0
            fulfilled_consents_consent_first_g = final_agent_values[consent_first_mask]['Number of Consents as G Fulfilled'].mean() if consent_first_mask.any() else 0
            fulfilled_consents_monitoring_r = final_agent_values[monitoring_mask]['Number of Consents as R Fulfilled'].mean() if monitoring_mask.any() else 0
            fulfilled_consents_monitoring_g = final_agent_values[monitoring_mask]['Number of Consents as G Fulfilled'].mean() if monitoring_mask.any() else 0
            
            # Calculate ratios (avoid division by zero)
            avg_consent_violation_ratio_consent_first_r = (violated_consents_consent_first_r / total_consents_consent_first_r) if total_consents_consent_first_r > 0 else 0
            avg_consent_violation_ratio_consent_first_g = (violated_consents_consent_first_g / total_consents_consent_first_g) if total_consents_consent_first_g > 0 else 0
            avg_consent_violation_ratio_monitoring_r = (violated_consents_monitoring_r / total_consents_monitoring_r) if total_consents_monitoring_r > 0 else 0
            avg_consent_violation_ratio_monitoring_g = (violated_consents_monitoring_g / total_consents_monitoring_g) if total_consents_monitoring_g > 0 else 0
            
            avg_consent_fulfillment_ratio_consent_first_r = (fulfilled_consents_consent_first_r / total_consents_consent_first_r) if total_consents_consent_first_r > 0 else 0
            avg_consent_fulfillment_ratio_consent_first_g = (fulfilled_consents_consent_first_g / total_consents_consent_first_g) if total_consents_consent_first_g > 0 else 0
            avg_consent_fulfillment_ratio_monitoring_r = (fulfilled_consents_monitoring_r / total_consents_monitoring_r) if total_consents_monitoring_r > 0 else 0
            avg_consent_fulfillment_ratio_monitoring_g = (fulfilled_consents_monitoring_g / total_consents_monitoring_g) if total_consents_monitoring_g > 0 else 0
        
            
            # Resource conflict counter goal accomplishment ratio
            avg_resource_conflict_counter_goal_accomplishment_ratio_consent_first_agent = (avg_resource_conflicts_consent_first_agent / avg_counter_goal_accomplishments_consent_first_agent) if avg_counter_goal_accomplishments_consent_first_agent > 0 else 0
            avg_resource_conflict_counter_goal_accomplishment_ratio_monitoring_agent = (avg_resource_conflicts_monitoring_agent / avg_counter_goal_accomplishments_monitoring_agent) if avg_counter_goal_accomplishments_monitoring_agent > 0 else 0
            
            # Calculate interaction and timing metrics
            avg_finished_step_consent_first_agent = final_agent_values[consent_first_mask]['Finished Step'].mean() if consent_first_mask.any() else 0
            avg_finished_step_monitoring_agent = final_agent_values[monitoring_mask]['Finished Step'].mean() if monitoring_mask.any() else 0
            avg_longest_idle_time_consent_first_agent = final_agent_values[consent_first_mask]['Longest Idle Time'].mean() if consent_first_mask.any() else 0
            avg_longest_idle_time_monitoring_agent = final_agent_values[monitoring_mask]['Longest Idle Time'].mean() if monitoring_mask.any() else 0
            avg_distinct_agents_interacted_r_consent_first_agent = final_agent_values[consent_first_mask]['Number of Distinct Agents Interacted as R'].mean() if consent_first_mask.any() else 0
            avg_distinct_agents_interacted_r_monitoring_agent = final_agent_values[monitoring_mask]['Number of Distinct Agents Interacted as R'].mean() if monitoring_mask.any() else 0
            avg_distinct_agents_interacted_g_consent_first_agent = final_agent_values[consent_first_mask]['Number of Distinct Agents Interacted as G'].mean() if consent_first_mask.any() else 0
            avg_distinct_agents_interacted_g_monitoring_agent = final_agent_values[monitoring_mask]['Number of Distinct Agents Interacted as G'].mean() if monitoring_mask.any() else 0
            # New: total idle time per agent
            avg_total_idle_time_consent_first_agent = final_agent_values[consent_first_mask]['Total Idle Time'].mean() if consent_first_mask.any() else 0
            avg_total_idle_time_monitoring_agent = final_agent_values[monitoring_mask]['Total Idle Time'].mean() if monitoring_mask.any() else 0
            
            # Calculate steps for this run as the last value of the Step/index column
            if not model_df.empty:
                if 'Step' in model_df.columns:
                    avg_steps_overall = int(pd.to_numeric(model_df['Step'], errors='coerce').dropna().iloc[-1])
                else:
                    first_col = model_df.columns[0]
                    avg_steps_overall = int(pd.to_numeric(model_df[first_col], errors='coerce').dropna().iloc[-1])
            else:
                avg_steps_overall = np.nan
            final_values = model_df.iloc[-1]
            
            simulation_data.append({
                'experiment_name': exp_name,
                'agent_config': agent_config,
                'seed': seed,
                'config_name': config_name,
                'consent_first_count': consent_first,
                'monitoring_count': monitoring,
                'fifty_fifty_count': fifty_fifty,
                'total_agents': total_agents,
                'accomplished_goals': final_values['Total Accomplished Goals'],
                'remaining_goals': final_values['Total Remaining Goals'],
                'violated_consents': final_values['Total Violated Consents'],
                'total_consents': final_values['Total Consent Activations'],
                'resource_conflicts': final_values['Total Resource Conflicts'],
                'counter_goal_accomplishments': final_values['Total Resource Conflict Accomplished Counter Goals'],
                'consent_violation_ratio': final_values['Consent Violation Ratio'],
                'consent_fulfillment_ratio': final_values['Consent Fulfillment Ratio'],
                'consent_unrealized_ratio': final_values['Consent Unrealized Ratio'],
                'consent_deferred_ratio': final_values['Consent Deferred Ratio'],
                'resource_conflict_counter_goal_accomplishment_ratio': final_values['Resource Conflict Counter Goal Accomplishment Ratio'],
                'max_steps': config.get('max_steps', 1000),
                'avg_steps_overall': avg_steps_overall,
                'avg_accomplished_goals_consent_first_agent': avg_accomplished_goals_consent_first_agent,
                'avg_accomplished_goals_monitoring_agent': avg_accomplished_goals_monitoring_agent,
                'avg_remaining_goals_consent_first_agent': avg_remaining_goals_consent_first_agent,
                'avg_remaining_goals_monitoring_agent': avg_remaining_goals_monitoring_agent,
                # R (Receiver) specific metrics
                'avg_total_consents_consent_first_r': total_consents_consent_first_r,
                'avg_total_consents_monitoring_r': total_consents_monitoring_r,
                'avg_violated_consents_consent_first_r': violated_consents_consent_first_r,
                'avg_violated_consents_monitoring_r': violated_consents_monitoring_r,
                'avg_fulfilled_consents_consent_first_r': fulfilled_consents_consent_first_r,
                'avg_fulfilled_consents_monitoring_r': fulfilled_consents_monitoring_r,
                'avg_consent_violation_ratio_consent_first_r': avg_consent_violation_ratio_consent_first_r,
                'avg_consent_violation_ratio_monitoring_r': avg_consent_violation_ratio_monitoring_r,
                'avg_consent_fulfillment_ratio_consent_first_r': avg_consent_fulfillment_ratio_consent_first_r,
                'avg_consent_fulfillment_ratio_monitoring_r': avg_consent_fulfillment_ratio_monitoring_r,
                
                # G (Giver) specific metrics
                'avg_total_consents_consent_first_g': total_consents_consent_first_g,
                'avg_total_consents_monitoring_g': total_consents_monitoring_g,
                'avg_violated_consents_consent_first_g': violated_consents_consent_first_g,
                'avg_violated_consents_monitoring_g': violated_consents_monitoring_g,
                'avg_fulfilled_consents_consent_first_g': fulfilled_consents_consent_first_g,
                'avg_fulfilled_consents_monitoring_g': fulfilled_consents_monitoring_g,
                'avg_consent_violation_ratio_consent_first_g': avg_consent_violation_ratio_consent_first_g,
                'avg_consent_violation_ratio_monitoring_g': avg_consent_violation_ratio_monitoring_g,
                'avg_consent_fulfillment_ratio_consent_first_g': avg_consent_fulfillment_ratio_consent_first_g,
                'avg_consent_fulfillment_ratio_monitoring_g': avg_consent_fulfillment_ratio_monitoring_g,
                
                # General agent metrics
                'avg_resource_conflicts_consent_first_agent': avg_resource_conflicts_consent_first_agent,
                'avg_resource_conflicts_monitoring_agent': avg_resource_conflicts_monitoring_agent,
                'avg_counter_goal_accomplishments_consent_first_agent': avg_counter_goal_accomplishments_consent_first_agent,
                'avg_counter_goal_accomplishments_monitoring_agent': avg_counter_goal_accomplishments_monitoring_agent,
                'avg_resource_conflict_counter_goal_accomplishment_ratio_consent_first_agent': avg_resource_conflict_counter_goal_accomplishment_ratio_consent_first_agent,
                'avg_resource_conflict_counter_goal_accomplishment_ratio_monitoring_agent': avg_resource_conflict_counter_goal_accomplishment_ratio_monitoring_agent,
                
                # Interaction and timing metrics
                'avg_finished_step_consent_first_agent': avg_finished_step_consent_first_agent,
                'avg_finished_step_monitoring_agent': avg_finished_step_monitoring_agent,
                'avg_longest_idle_time_consent_first_agent': avg_longest_idle_time_consent_first_agent,
                'avg_longest_idle_time_monitoring_agent': avg_longest_idle_time_monitoring_agent,
                'avg_distinct_agents_interacted_r_consent_first_agent': avg_distinct_agents_interacted_r_consent_first_agent,
                'avg_distinct_agents_interacted_r_monitoring_agent': avg_distinct_agents_interacted_r_monitoring_agent,
                'avg_distinct_agents_interacted_g_consent_first_agent': avg_distinct_agents_interacted_g_consent_first_agent,
                'avg_distinct_agents_interacted_g_monitoring_agent': avg_distinct_agents_interacted_g_monitoring_agent,
                # New: total idle time
                'avg_total_idle_time_consent_first_agent': avg_total_idle_time_consent_first_agent,
                'avg_total_idle_time_monitoring_agent': avg_total_idle_time_monitoring_agent,
            })
        else:
            print(f"Warning: Model data file not found for {config_name}")
    
    return pd.DataFrame(simulation_data), timestamp_date

def create_agent_ratio_analysis(experiment_name=None, experiment_date=None):
    """Create comprehensive analysis of how metrics change with agent ratios.
    
    This function averages results across all seeds for each experiment configuration.
    """
    print(f"Analyzing experiment: {experiment_name}, date: {experiment_date}")
    
    # Create figures directory
    figures_dir = create_figures_directory(experiment_name, experiment_date)
    
    # Load data
    df, timestamp_date = load_simulation_data(experiment_name=experiment_name, experiment_date=experiment_date)
    
    if df.empty:
        print("No simulation data found!")
        return
    
    # Group by experiment_name and agent_config, then calculate mean and std
    metrics_to_average = [
        'consent_first_count', 'monitoring_count', 'fifty_fifty_count', 'total_agents',
        'accomplished_goals', 'remaining_goals', 'violated_consents', 'total_consents',
        'resource_conflicts', 'counter_goal_accomplishments',
        'consent_violation_ratio', 'consent_fulfillment_ratio', 
        'consent_unrealized_ratio', 'consent_deferred_ratio',
        'resource_conflict_counter_goal_accomplishment_ratio', 'avg_steps_overall'
    ]
    
    # Calculate mean and standard error for each metric
    grouped = df.groupby(['experiment_name', 'agent_config'])
    
    mean_df = grouped[metrics_to_average].mean().reset_index()

    return mean_df, df

mean_df, df = create_agent_ratio_analysis(experiment_name="monitoring_vs_consent_based_analysis", experiment_date="20251108")


Analyzing experiment: monitoring_vs_consent_based_analysis, date: 20251108
Found experiment-specific configs in: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/monitoring_vs_consent_based_analysis_20251108/configs
Found matching configs in subdirectory: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/monitoring_vs_consent_based_analysis_20251108/configs (110 files)


## Graph 01-02: 1-way ANOVA: Accomplished Goals


In [3]:
df_sorted = df.drop_duplicates().sort_values(by=['monitoring_count', 'seed'], ascending=True)
df_sorted["consent_violation_ratio"] = df_sorted["violated_consents"] / df_sorted["total_consents"]
groups = [
    df_sorted[df_sorted["monitoring_count"] == r]["accomplished_goals"]
    for r in sorted(df_sorted["monitoring_count"].unique())
]

# One-way ANOVA
F, p = stats.f_oneway(*groups)

# Effect size: eta-squared for one-way ANOVA
k = len(groups)                             # number of groups
ns = [len(g) for g in groups]
N = sum(ns)                                 # total sample size
df_between = k - 1
df_within = N - k
eta_sq = (F * df_between) / (F * df_between + df_within)

# Get min / max values of the averages graph
max_accomplished_goals = mean_df.groupby("monitoring_count")["accomplished_goals"].mean().max()
min_accomplished_goals = mean_df.groupby("monitoring_count")["accomplished_goals"].mean().min()

print(f"F: {F:.4f}, p: {p:.3e}")
print(f"Eta-squared (effect size): {eta_sq:.4f}")
print(f"Max: {max_accomplished_goals}, Min: {min_accomplished_goals}")


F: 2.1735, p: 2.550e-02
Eta-squared (effect size): 0.1800
Max: 3000.0, Min: 2997.7


## Graph 03: 1-way ANOVA: Consent Violation Ratio


In [4]:
groups = [
    df_sorted[df_sorted["monitoring_count"] == r]["consent_violation_ratio"]
    for r in sorted(df_sorted["monitoring_count"].unique())
]

# One-way ANOVA for consent violation ratios
F, p = stats.f_oneway(*groups)

# Effect size: eta-squared for one-way ANOVA
k = len(groups)                             # number of groups
ns = [len(g) for g in groups]
N = sum(ns)                                 # total sample size
df_between = k - 1
df_within = N - k
eta_sq = (F * df_between) / (F * df_between + df_within)

max_consent_violation_ratio = mean_df.groupby("monitoring_count")["consent_violation_ratio"].mean().max()
min_consent_violation_ratio = mean_df.groupby("monitoring_count")["consent_violation_ratio"].mean().min()

print(f"F: {F:.4f}, p: {p:.3e}")
print(f"Eta-squared (effect size): {eta_sq:.4f}")
print(f"Max: {max_consent_violation_ratio}, Min: {min_consent_violation_ratio}")


F: 312.7200, p: 3.420e-70
Eta-squared (effect size): 0.9693
Max: 0.4099140920086988, Min: 0.1549459627001371


## Get Agent Level Data


In [5]:
def create_agent_level_analysis(experiment_name=None, experiment_date=None):
    """Create analysis of agent-level metrics comparing ConsentFirstAgent and MonitoringAgent.
    
    This function shows how individual agent performance varies across different configurations.
    Uses agent CSV files and config JSON files directly, no model CSV files.
    """
    print(f"\nCreating Agent-Level Analysis for: {experiment_name}, date: {experiment_date}")
    
    # Create figures directory
    figures_dir = create_figures_directory(experiment_name, experiment_date)
    print(f"Figures will be saved to: {figures_dir}")
    
    results_dir = Path("/Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results")
    
    def _find_agent_file(prefix: str):
        main_data = results_dir / "data"
        p = main_data / f"{prefix}_agents.csv"
        if p.exists():
            return p
        for sub in results_dir.iterdir():
            if sub.is_dir():
                d = sub / "data"
                q = d / f"{prefix}_agents.csv"
                if d.exists() and q.exists():
                    return q
        return None
    
    def _find_model_file(prefix: str):
        main_data = results_dir / "data"
        p = main_data / f"{prefix}_model.csv"
        if p.exists():
            return p
        for sub in results_dir.iterdir():
            if sub.is_dir():
                d = sub / "data"
                q = d / f"{prefix}_model.csv"
                if d.exists() and q.exists():
                    return q
        return None
    
    # Find all config files - check both main directories and experiment-specific subdirectories
    config_files = []
    
    # Check main configs directory
    main_configs_dir = results_dir / "configs"
    if main_configs_dir.exists():
        config_files.extend(list(main_configs_dir.glob("*.json")))
    
    # Check experiment-specific subdirectories
    if experiment_name and experiment_date:
        exp_subdir = results_dir / f"{experiment_name}_{experiment_date}"
        if exp_subdir.exists():
            exp_configs_dir = exp_subdir / "configs"
            if exp_configs_dir.exists():
                config_files.extend(list(exp_configs_dir.glob("*.json")))
                print(f"Found experiment-specific configs in: {exp_configs_dir}")
    
    # Also search all subdirectories for files that match the experiment name and date
    if experiment_name and experiment_date:
        for subdir in results_dir.iterdir():
            if subdir.is_dir():
                exp_configs_dir = subdir / "configs"
                if exp_configs_dir.exists():
                    # Check if any files in this directory match our criteria
                    matching_files = []
                    for config_file in exp_configs_dir.glob("*.json"):
                        exp_name, agent_config, seed, file_date = extract_experiment_info(config_file.name)
                        if exp_name == experiment_name and file_date == experiment_date:
                            matching_files.append(config_file)
                    
                    if matching_files:
                        config_files.extend(matching_files)
                        print(f"Found matching configs in subdirectory: {exp_configs_dir} ({len(matching_files)} files)")
    
    # Collect agent-level data from all agent CSV files
    agent_data_list = []
    agent_data_list_all_steps = []
    
    for config_file in config_files:
        # Extract experiment info from filename
        exp_name, agent_config, seed, timestamp_date = extract_experiment_info(config_file.name)
        
        # Skip if we can't parse the filename or if it doesn't match the desired experiment
        if exp_name is None:
            print(f"Warning: Could not parse filename: {config_file.name}")
            continue
        
        if experiment_name is not None and exp_name != experiment_name:
            continue
        
        # Filter by experiment date if specified
        if experiment_date is not None and timestamp_date != experiment_date:
            continue
        
        # Load config to get agent counts
        try:
            with open(config_file, 'r') as f:
                config = json.load(f)
        except Exception as e:
            print(f"Warning: Could not load config file {config_file}: {e}")
            continue
        
        # Extract agent counts from config
        params = config.get('parameters', {})
        consent_first = params.get('ConsentFirstAgent_COUNT', 0)
        monitoring = params.get('MonitoringAgent_COUNT', 0)
        fifty_fifty = params.get('FiftyFiftyAgent_COUNT', 0)
        total_agents = consent_first + monitoring + fifty_fifty
        
        # Get config name (without _config.json suffix)
        config_name = config_file.stem
        
        # Find corresponding agent file
        prefix = config_name.rsplit('_', 1)[0]
        agent_file = _find_agent_file(prefix)
        model_file = _find_model_file(prefix)
        
        if agent_file is None or not agent_file.exists():
            print(f"Warning: Agent file not found for {prefix}")
            continue
        
        if model_file is None or not model_file.exists():
            print(f"Warning: Model file not found for {prefix}")
            continue
        
        try:
            agent_df = pd.read_csv(agent_file)
            model_df = pd.read_csv(model_file)
            
            # Get the step column
            step_col = 'Step' if 'Step' in agent_df.columns else agent_df.columns[0]

            early_stop_steps = config.get('early_stop_steps', 0)
            if early_stop_steps > 1:
                # Exclude the last (early_stop_steps - 1) rows
                steps_to_exclude = early_stop_steps - 1
                distinct_agent_count_q = """SELECT COUNT(DISTINCT AgentID) as AGENT_COUNT FROM agent_df WHERE Step = 1"""
                distinct_agent_count = ps.sqldf(distinct_agent_count_q, locals())["AGENT_COUNT"][0]

                if len(model_df) > steps_to_exclude and model_df.iloc[-steps_to_exclude]['Total Accomplished Goals'] == model_df.iloc[-1]['Total Accomplished Goals']:
                    agent_df = agent_df.iloc[:-steps_to_exclude*distinct_agent_count]
            
            # Get final step and calculate avg_steps_overall from agent CSV
            steps = pd.to_numeric(agent_df[step_col], errors='coerce')
            last_step = steps.max()
            avg_steps_overall = int(last_step) if not pd.isna(last_step) else np.nan
            
            final_agent_values = agent_df[steps == last_step].copy()
            
            if 'Agent Persona' not in final_agent_values.columns:
                print(f"Warning: 'Agent Persona' column not found in {agent_file}")
                continue
            
            # Create masks for agent types
            consent_first_mask = final_agent_values['Agent Persona'] == 'ConsentFirstAgent'
            monitoring_mask = final_agent_values['Agent Persona'] == 'MonitoringAgent'

            final_agent_values["seed"] = seed
            final_agent_values["agent_config"] = agent_config

            agent_df["seed"] = seed
            agent_df["agent_config"] = agent_config
            
            agent_data_list.append(final_agent_values)
            agent_data_list_all_steps.append(agent_df)
        except Exception as e:
            print(f"Error processing {prefix}: {e}")
            import traceback
            traceback.print_exc()
            continue

    if len(agent_data_list) == 0:
        print("Warning: No agent data collected. Returning None.")
        return None
    
    all_agent_values_df = pd.concat(agent_data_list)
    # Extract monitoring_count from agent_config (format: "1000-0-0-0" where last number is monitoring)
    all_agent_values_df["monitoring_count"] = all_agent_values_df["agent_config"].str.split("-").str[-1].astype(int)

    all_agent_values_df_all_steps = pd.concat(agent_data_list_all_steps)
    all_agent_values_df_all_steps["monitoring_count"] = all_agent_values_df_all_steps["agent_config"].str.split("-").str[-1].astype(int)
    return all_agent_values_df, all_agent_values_df_all_steps


In [6]:
final_agent_values, all_agent_values_df_all_steps = create_agent_level_analysis(experiment_name="monitoring_vs_consent_based_analysis", experiment_date="20251119")
#agent_mean_df
final_agent_values



Creating Agent-Level Analysis for: monitoring_vs_consent_based_analysis, date: 20251119
Figures will be saved to: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/monitoring_vs_consent_based_analysis_20251119/figures
Found experiment-specific configs in: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/monitoring_vs_consent_based_analysis_20251119/configs
Found matching configs in subdirectory: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/monitoring_vs_consent_based_analysis_20251119/configs (110 files)


/var/folders/qz/c3nhpt0n4_7g6vm6v12rwpxm0000gn/T/ipykernel_65254/3673815334.py:161: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  agent_df["seed"] = seed
/var/folders/qz/c3nhpt0n4_7g6vm6v12rwpxm0000gn/T/ipykernel_65254/3673815334.py:162: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  agent_df["agent_config"] = agent_config
/var/folders/qz/c3nhpt0n4_7g6vm6v12rwpxm0000gn/T/ipykernel_65254/3673815334.py:161: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

,Step,AgentID,Agent Persona,Accomplished Goals,Remaining Goals,Resource Conflicts,Counter Conflict Goal Accomplishments,Finished Step,Longest Idle Time,Total Idle Time,...,Number of Consents as G,Number of Consents as R Violated,Number of Consents as R Fulfilled,Number of Consents as R Unrealized,Number of Consents as G Violated,Number of Consents as G Fulfilled,Number of Consents as G Unrealized,seed,agent_config,monitoring_count
24000,24,1,ConsentFirstAgent,3,0,0,0,14.0,10,11,...,21,6,9,0,2,18,1,789,300-0-0-700,700
24001,24,2,ConsentFirstAgent,3,0,4,3,16.0,13,13,...,14,10,8,0,1,11,1,789,300-0-0-700,700
24002,24,3,ConsentFirstAgent,3,0,15,0,21.0,18,18,...,12,14,8,0,4,8,0,789,300-0-0-700,700
24003,24,4,ConsentFirstAgent,3,0,14,0,19.0,13,16,...,32,5,8,0,5,24,3,789,300-0-0-700,700
24004,24,5,ConsentFirstAgent,3,0,0,0,12.0,6,9,...,14,2,10,0,3,11,0,789,300-0-0-700,700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23995,23,996,MonitoringAgent,3,0,6,2,13.0,9,10,...,9,2,8,1,2,6,1,35,200-0-0-800,800
23996,23,997,MonitoringAgent,3,0,1,0,18.0,14,15,...,11,5,9,1,2,7,2,35,200-0-0-800,800
23997,23,998,MonitoringAgent,3,0,0,0,16.0,11,13,...,10,3,9,1,0,9,1,35,200-0-0-800,800
23998,23,999,MonitoringAgent,3,0,4,1,17.0,13,14,...,9,6,9,2,1,5,2,35,200-0-0-800,800


In [7]:
all_agent_values_df_all_steps[["seed", "agent_config", "monitoring_count", "Step",  "AgentID", "Agent Persona", "Accomplished Goals"]]


,seed,agent_config,monitoring_count,Step,AgentID,Agent Persona,Accomplished Goals
0,789,300-0-0-700,700,0,1,ConsentFirstAgent,0
1,789,300-0-0-700,700,0,2,ConsentFirstAgent,0
2,789,300-0-0-700,700,0,3,ConsentFirstAgent,0
3,789,300-0-0-700,700,0,4,ConsentFirstAgent,0
4,789,300-0-0-700,700,0,5,ConsentFirstAgent,0
...,...,...,...,...,...,...,...
23995,35,200-0-0-800,800,23,996,MonitoringAgent,3
23996,35,200-0-0-800,800,23,997,MonitoringAgent,3
23997,35,200-0-0-800,800,23,998,MonitoringAgent,3
23998,35,200-0-0-800,800,23,999,MonitoringAgent,3


In [8]:
df_sorted = final_agent_values.copy()

# Compute agent-level ratios
df_sorted = df_sorted[df_sorted["Number of Consents as R"] > 0]
df_sorted["consent_violation_ratio"] = (
    df_sorted["Number of Consents as R Violated"] /
    df_sorted["Number of Consents as R"]
)

df_sorted["consent_fulfillment_ratio"] = (
    df_sorted["Number of Consents as R Fulfilled"] /
    df_sorted["Number of Consents as R"]
)

df_sorted["consent_unrealized_ratio"] = (
    df_sorted["Number of Consents as R Unrealized"] /
    df_sorted["Number of Consents as R"]
)

df_sorted["tot_idle_time_normalized"] = df_sorted["Total Idle Time"] / df_sorted["Step"]

# ---- AGGREGATE per simulation run ----
run_level_df = (
    df_sorted.groupby(["monitoring_count", "seed", "Agent Persona"])[
        ["consent_violation_ratio", "consent_fulfillment_ratio", "consent_unrealized_ratio", "Accomplished Goals", "tot_idle_time_normalized"]
    ]
    .mean()
    .reset_index()
)

df_da = run_level_df[run_level_df["Agent Persona"] == "ConsentFirstAgent"]
df_va = run_level_df[run_level_df["Agent Persona"] == "MonitoringAgent"]

# Create separate dataframes for "Accomplished Goals" and "tot_idle_time_normalized" 
# that include ALL agents (not just those with consents)
df_all_agents = final_agent_values.copy()
df_all_agents["tot_idle_time_normalized"] = df_all_agents["Total Idle Time"] / df_all_agents["Step"]

run_level_df_all_agents = (
    df_all_agents.groupby(["monitoring_count", "seed", "Agent Persona"])[
        ["Accomplished Goals", "tot_idle_time_normalized", "Total Idle Time"]
    ]
    .mean()
    .reset_index()
)

df_da_full = run_level_df_all_agents[run_level_df_all_agents["Agent Persona"] == "ConsentFirstAgent"]
df_va_full = run_level_df_all_agents[run_level_df_all_agents["Agent Persona"] == "MonitoringAgent"]


## 04 Consent Violation Ratio, 1-WAY ANOVA TESTS


In [9]:
# One-way ANOVA for ConsentFirstAgent (DA)
groups_da = [
    df_da[df_da["monitoring_count"] == r]["consent_violation_ratio"]
    for r in sorted(df_da["monitoring_count"].unique())
]

F_da, p_da = stats.f_oneway(*groups_da)

# Effect size (eta-squared) for DA
k_da = len(groups_da)                       # number of groups
ns_da = [len(g) for g in groups_da]
N_da = sum(ns_da)                          # total sample size
df_between_da = k_da - 1
df_within_da = N_da - k_da
eta_sq_da = (F_da * df_between_da) / (F_da * df_between_da + df_within_da)

# Get min / max values of the averages
max_consent_violation_ratio_da = df_da.groupby("monitoring_count")["consent_violation_ratio"].mean().max()
min_consent_violation_ratio_da = df_da.groupby("monitoring_count")["consent_violation_ratio"].mean().min()

print(f"ConsentFirstAgent (DA) - F: {F_da:.4f}, p: {p_da:.3e}, eta^2: {eta_sq_da:.4f}")
print(f"  Max: {max_consent_violation_ratio_da:.4f}, Min: {min_consent_violation_ratio_da:.4f}")

# One-way ANOVA for MonitoringAgent (VA)
groups_va = [
    df_va[df_va["monitoring_count"] == r]["consent_violation_ratio"]
    for r in sorted(df_va["monitoring_count"].unique())
]

F_va, p_va = stats.f_oneway(*groups_va)

# Effect size (eta-squared) for VA
k_va = len(groups_va)                       # number of groups
ns_va = [len(g) for g in groups_va]
N_va = sum(ns_va)                          # total sample size
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va = (F_va * df_between_va) / (F_va * df_between_va + df_within_va)

# Get min / max values of the averages
max_consent_violation_ratio_va = df_va.groupby("monitoring_count")["consent_violation_ratio"].mean().max()
min_consent_violation_ratio_va = df_va.groupby("monitoring_count")["consent_violation_ratio"].mean().min()

print(f"MonitoringAgent (VA) - F: {F_va:.4f}, p: {p_va:.3e}, eta^2: {eta_sq_va:.4f}")
print(f"  Max: {max_consent_violation_ratio_va:.4f}, Min: {min_consent_violation_ratio_va:.4f}")


ConsentFirstAgent (DA) - F: 10.7165, p: 3.887e-11, eta^2: 0.5173
  Max: 0.3692, Min: 0.3117
MonitoringAgent (VA) - F: 35.8583, p: 4.634e-26, eta^2: 0.7819
  Max: 0.2246, Min: 0.1656


## 05 Consent Fulfilment Ratio, 1-WAY ANOVA TESTS


In [10]:
# One-way ANOVA for ConsentFirstAgent (DA) - Fulfilment Ratio
groups_da = [
    df_da[df_da["monitoring_count"] == r]["consent_fulfillment_ratio"]
    for r in sorted(df_da["monitoring_count"].unique())
]

F_da, p_da = stats.f_oneway(*groups_da)

# Effect size (eta-squared) for DA
k_da = len(groups_da)                       # number of groups
ns_da = [len(g) for g in groups_da]
N_da = sum(ns_da)                          # total sample size
df_between_da = k_da - 1
df_within_da = N_da - k_da
eta_sq_da = (F_da * df_between_da) / (F_da * df_between_da + df_within_da)

# Get min / max values of the averages
max_consent_fulfillment_ratio_da = df_da.groupby("monitoring_count")["consent_fulfillment_ratio"].mean().max()
min_consent_fulfillment_ratio_da = df_da.groupby("monitoring_count")["consent_fulfillment_ratio"].mean().min()

print(f"ConsentFirstAgent (DA) - F: {F_da:.4f}, p: {p_da:.3e}, eta^2: {eta_sq_da:.4f}")
print(f"  Max: {max_consent_fulfillment_ratio_da:.4f}, Min: {min_consent_fulfillment_ratio_da:.4f}")

# One-way ANOVA for MonitoringAgent (VA) - Fulfilment Ratio
groups_va = [
    df_va[df_va["monitoring_count"] == r]["consent_fulfillment_ratio"]
    for r in sorted(df_va["monitoring_count"].unique())
]

F_va, p_va = stats.f_oneway(*groups_va)

# Effect size (eta-squared) for VA
k_va = len(groups_va)                       # number of groups
ns_va = [len(g) for g in groups_va]
N_va = sum(ns_va)                          # total sample size
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va = (F_va * df_between_va) / (F_va * df_between_va + df_within_va)

# Get min / max values of the averages
max_consent_fulfillment_ratio_va = df_va.groupby("monitoring_count")["consent_fulfillment_ratio"].mean().max()
min_consent_fulfillment_ratio_va = df_va.groupby("monitoring_count")["consent_fulfillment_ratio"].mean().min()

print(f"MonitoringAgent (VA) - F: {F_va:.4f}, p: {p_va:.3e}, eta^2: {eta_sq_va:.4f}")
print(f"  Max: {max_consent_fulfillment_ratio_va:.4f}, Min: {min_consent_fulfillment_ratio_va:.4f}")


ConsentFirstAgent (DA) - F: 10.7657, p: 3.519e-11, eta^2: 0.5184
  Max: 0.6883, Min: 0.6306
MonitoringAgent (VA) - F: 26.4486, p: 1.106e-21, eta^2: 0.7256
  Max: 0.7105, Min: 0.6372


## 06: Consent Unrealized Ratio, 1-WAY ANOVA TESTS



In [11]:
# One-way ANOVA for ConsentFirstAgent (DA) - Unrealized Ratio
groups_da = [
    df_da[df_da["monitoring_count"] == r]["consent_unrealized_ratio"]
    for r in sorted(df_da["monitoring_count"].unique())
]

F_da, p_da = stats.f_oneway(*groups_da)

# Effect size (eta-squared) for DA
k_da = len(groups_da)                       # number of groups
ns_da = [len(g) for g in groups_da]
N_da = sum(ns_da)                          # total sample size
df_between_da = k_da - 1
df_within_da = N_da - k_da
eta_sq_da = (F_da * df_between_da) / (F_da * df_between_da + df_within_da)

# Get min / max values of the averages
max_consent_unrealized_ratio_da = df_da.groupby("monitoring_count")["consent_unrealized_ratio"].mean().max()
min_consent_unrealized_ratio_da = df_da.groupby("monitoring_count")["consent_unrealized_ratio"].mean().min()

print(f"ConsentFirstAgent (DA) - F: {F_da:.4f}, p: {p_da:.3e}, eta^2: {eta_sq_da:.4f}")
print(f"  Max: {max_consent_unrealized_ratio_da:.4f}, Min: {min_consent_unrealized_ratio_da:.4f}")

# One-way ANOVA for MonitoringAgent (VA) - Unrealized Ratio
groups_va = [
    df_va[df_va["monitoring_count"] == r]["consent_unrealized_ratio"]
    for r in sorted(df_va["monitoring_count"].unique())
]

F_va, p_va = stats.f_oneway(*groups_va)

# Effect size (eta-squared) for VA
k_va = len(groups_va)                       # number of groups
ns_va = [len(g) for g in groups_va]
N_va = sum(ns_va)                          # total sample size
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va = (F_va * df_between_va) / (F_va * df_between_va + df_within_va)

# Get min / max values of the averages
max_consent_unrealized_ratio_va = df_va.groupby("monitoring_count")["consent_unrealized_ratio"].mean().max()
min_consent_unrealized_ratio_va = df_va.groupby("monitoring_count")["consent_unrealized_ratio"].mean().min()

print(f"MonitoringAgent (VA) - F: {F_va:.4f}, p: {p_va:.3e}, eta^2: {eta_sq_va:.4f}")
print(f"  Max: {max_consent_unrealized_ratio_va:.4f}, Min: {min_consent_unrealized_ratio_va:.4f}")



ConsentFirstAgent (DA) - F: nan, p: nan, eta^2: nan
  Max: 0.0000, Min: 0.0000
MonitoringAgent (VA) - F: 4.6066, p: 5.124e-05, eta^2: 0.3154
  Max: 0.0916, Min: 0.0792


/Users/efeonal/py_envs/MESA_thesis/venv/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, **kwds)


## 04: Consent Violation Ratio Mann-Whitney Test


In [12]:
import pandas as pd
from scipy.stats import mannwhitneyu

# ----------------------------------------------------------------------
# 1. Prepare data properly: compute ratios + aggregate per run
# ----------------------------------------------------------------------

df_sorted = final_agent_values.copy()

# Keep only agents with at least 1 consent as R
df_sorted = df_sorted[df_sorted["Number of Consents as R"] > 0]

# Compute ratios
df_sorted["consent_violation_ratio"] = (
    df_sorted["Number of Consents as R Violated"] /
    df_sorted["Number of Consents as R"]
)

df_sorted["consent_fulfillment_ratio"] = (
    df_sorted["Number of Consents as R Fulfilled"] /
    df_sorted["Number of Consents as R"]
)

run_level_df = df_sorted.groupby(["monitoring_count", "seed", "Agent Persona"])[[
    "Number of Consents as R",
    "Number of Consents as R Violated",
    "Number of Consents as R Fulfilled"
]].sum().reset_index()

# Aggregate so that each (ratio × seed × persona) is one data point
run_level_df["consent_violation_ratio"] = run_level_df["Number of Consents as R Violated"] / run_level_df["Number of Consents as R"]
run_level_df["consent_fulfillment_ratio"] = run_level_df["Number of Consents as R Fulfilled"] / run_level_df["Number of Consents as R"]


# Split by persona
df_da = run_level_df[run_level_df["Agent Persona"] == "ConsentFirstAgent"]
df_va = run_level_df[run_level_df["Agent Persona"] == "MonitoringAgent"]

# ----------------------------------------------------------------------
# 2. Compare violation ratios between personas
# ----------------------------------------------------------------------

# Means per condition (align conditions across personas)
consent_first_means = df_da.groupby("monitoring_count")["consent_violation_ratio"].mean()
monitoring_means = df_va.groupby("monitoring_count")["consent_violation_ratio"].mean()

# Use union of all observed monitoring_count values and fill missing persona means with 0
all_counts = sorted(set(consent_first_means.index).union(set(monitoring_means.index)))
consent_first_aligned = consent_first_means.reindex(all_counts, fill_value=0)
monitoring_aligned = monitoring_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "monitoring_count": all_counts,
    "ConsentFirstAgent_mean": consent_first_aligned.values,
    "MonitoringAgent_mean": monitoring_aligned.values
})
comparison_df["Difference"] = (
    comparison_df["ConsentFirstAgent_mean"] -
    comparison_df["MonitoringAgent_mean"]
)
comparison_df["ConsentFirst_higher"] = comparison_df["Difference"] > 0

print("\nViolation Ratio Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

# Count conditions
consent_first_higher_count = comparison_df["ConsentFirst_higher"].sum()
monitoring_higher_count = len(comparison_df) - consent_first_higher_count

print(f"\nConsentFirstAgent higher in {consent_first_higher_count}/"
      f"{len(comparison_df)} conditions")
print(f"MonitoringAgent higher in {monitoring_higher_count}/"
      f"{len(comparison_df)} conditions")

# Overall means using run-level data
overall_da = df_da["consent_violation_ratio"].mean()
overall_va = df_va["consent_violation_ratio"].mean()

print("\nOverall mean violation ratios:")
print(f"  ConsentFirstAgent: {overall_da:.4f}")
print(f"  MonitoringAgent:    {overall_va:.4f}")
print(f"  Difference:         {overall_da - overall_va:.4f}")

# Mann–Whitney U tests
print("\nMann–Whitney U Test (Violation Ratios):")
cons_da = df_da["consent_violation_ratio"]
cons_va = df_va["consent_violation_ratio"]

u_twosided, p_twosided = mannwhitneyu(cons_da, cons_va, alternative="two-sided")
u_da_greater, p_da_greater = mannwhitneyu(cons_da, cons_va, alternative="greater")
u_va_greater, p_va_greater = mannwhitneyu(cons_va, cons_da, alternative="greater")

print(f"  Two-sided:           p = {p_twosided:.3e}")
print(f"  DA > VA (one-sided): p = {p_da_greater:.3e}")
print(f"  VA > DA (one-sided): p = {p_va_greater:.3e}")

# Rank-biserial correlation (effect size) using two-sided U
n_da = len(cons_da)
n_va = len(cons_va)
if n_da > 0 and n_va > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_da * n_va)
    # Give direction based on overall means
    direction = 1 if overall_da > overall_va else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if overall_da > overall_va:
        print("  → SIGNIFICANT: ConsentFirstAgent violate more.")
    else:
        print("  → SIGNIFICANT: MonitoringAgent violate more.")
else:
    print("  → NO significant difference.")




Violation Ratio Comparison by Persona:
 monitoring_count  ConsentFirstAgent_mean  MonitoringAgent_mean  Difference  ConsentFirst_higher
                0                0.409914              0.000000    0.409914                 True
              100                0.385585              0.251251    0.134335                 True
              200                0.373175              0.239415    0.133760                 True
              300                0.366053              0.230256    0.135797                 True
              400                0.363762              0.227720    0.136043                 True
              500                0.360586              0.219523    0.141062                 True
              600                0.359672              0.214711    0.144962                 True
              700                0.361214              0.206578    0.154636                 True
              800                0.359300              0.202663    0.156637            

## 05: Consent Fulfilment Ratio Mann-Whitney Test (Excluding Homogeneous Conditions)

In [13]:
# ----------------------------------------------------------------------
# 1. Prepare data properly: compute ratios + aggregate per run
# ----------------------------------------------------------------------

# ----------------------------------------------------------------------
# 2. Compare fulfilment ratios between personas across all monitoring counts
# ----------------------------------------------------------------------

# Mean per condition (align conditions across personas)
consent_first_means = df_da.groupby("monitoring_count")["consent_fulfillment_ratio"].mean()
monitoring_means    = df_va.groupby("monitoring_count")["consent_fulfillment_ratio"].mean()

# Use union of all observed monitoring_count values and fill missing persona means with 0
all_counts = sorted(set(consent_first_means.index).union(set(monitoring_means.index)))
consent_first_aligned = consent_first_means.reindex(all_counts, fill_value=0)
monitoring_aligned = monitoring_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "monitoring_count": all_counts,
    "ConsentFirstAgent_mean": consent_first_aligned.values,
    "MonitoringAgent_mean":    monitoring_aligned.values
})
comparison_df["Difference"] = (
    comparison_df["ConsentFirstAgent_mean"] -
    comparison_df["MonitoringAgent_mean"]
)
comparison_df["ConsentFirst_higher"] = comparison_df["Difference"] > 0

print("\nFulfillment Ratio Comparison by Persona (all monitoring counts):")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

consent_first_higher_count = comparison_df["ConsentFirst_higher"].sum()
monitoring_higher_count    = len(comparison_df) - consent_first_higher_count

print(f"\nConsentFirstAgent higher in {consent_first_higher_count}/"
      f"{len(comparison_df)} conditions")
print(f"MonitoringAgent   higher in {monitoring_higher_count}/"
      f"{len(comparison_df)} conditions")

# ----------------------------------------------------------------------
# 3. Mann–Whitney U tests (excluding homogeneous conditions)
#    Test on all distributions, excluding first and last (monitoring_count = 0 and 1000)
# ----------------------------------------------------------------------

# Exclude homogeneous conditions (first and last population distributions)
all_monitoring_counts = sorted(run_level_df["monitoring_count"].unique())
if len(all_monitoring_counts) > 2:
    min_monitoring_count = min(all_monitoring_counts)
    max_monitoring_count = max(all_monitoring_counts)
    # Filter out homogeneous conditions
    run_level_df_filtered = run_level_df[~run_level_df["monitoring_count"].isin([min_monitoring_count, max_monitoring_count])]
    print(f"\nNote: Excluding homogeneous conditions (monitoring_count = {min_monitoring_count} and {max_monitoring_count})")
else:
    run_level_df_filtered = run_level_df

# Split by persona for filtered data
df_da_filtered = run_level_df_filtered[run_level_df_filtered["Agent Persona"] == "ConsentFirstAgent"]
df_va_filtered = run_level_df_filtered[run_level_df_filtered["Agent Persona"] == "MonitoringAgent"]

# Calculate condition-level means (one value per condition per persona)
# This matches the graph structure where each point is a condition mean
da_condition_means = df_da_filtered.groupby("monitoring_count")["consent_fulfillment_ratio"].mean()
va_condition_means = df_va_filtered.groupby("monitoring_count")["consent_fulfillment_ratio"].mean()

# Align to common conditions (where both personas have data)
common_conditions = sorted(set(da_condition_means.index).intersection(set(va_condition_means.index)))
da_aligned = da_condition_means.loc[common_conditions]
va_aligned = va_condition_means.loc[common_conditions]

print(f"\nMann–Whitney U Test (Fulfillment Ratios, excluding homogeneous conditions):")
print("Note: Comparing condition-level means (averaged across runs per condition)")
print("      This matches the graph structure where each point is a condition mean")
print(f"\n  Number of conditions compared: {len(common_conditions)}")
print(f"  Conditions: {common_conditions}")

# Mann–Whitney U test on condition-level means
u_twosided, p_twosided = mannwhitneyu(da_aligned, va_aligned, alternative="two-sided")
u_da_greater, p_da_greater = mannwhitneyu(da_aligned, va_aligned, alternative="greater")
u_va_greater, p_va_greater = mannwhitneyu(va_aligned, da_aligned, alternative="greater")

mean_da_conditions = da_aligned.mean()
mean_va_conditions = va_aligned.mean()

print(f"\n  ConsentFirstAgent mean (condition-level): {mean_da_conditions:.4f}")
print(f"  MonitoringAgent   mean (condition-level): {mean_va_conditions:.4f}")
print(f"  Difference:       {mean_da_conditions - mean_va_conditions:.4f}")
print(f"\n  Two-sided:           p = {p_twosided:.3e}")
print(f"  DA > VA (one-sided): p = {p_da_greater:.3e}")
print(f"  VA > DA (one-sided): p = {p_va_greater:.3e}")

# Rank-biserial correlation (effect size) using two-sided U
n_da = len(da_aligned)
n_va = len(va_aligned)
if n_da > 0 and n_va > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_da * n_va)
    # Give direction based on condition-level means
    direction = 1 if mean_da_conditions > mean_va_conditions else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if mean_da_conditions > mean_va_conditions:
        print("  → SIGNIFICANT: ConsentFirstAgent fulfils more.")
    else:
        print("  → SIGNIFICANT: MonitoringAgent fulfils more.")
else:
    print("  → NO significant difference.")



Fulfillment Ratio Comparison by Persona (all monitoring counts):
 monitoring_count  ConsentFirstAgent_mean  MonitoringAgent_mean  Difference  ConsentFirst_higher
                0                0.589875              0.000000    0.589875                 True
              100                0.614415              0.600016    0.014399                 True
              200                0.626607              0.609196    0.017411                 True
              300                0.633883              0.620544    0.013340                 True
              400                0.636138              0.623818    0.012320                 True
              500                0.639414              0.632658    0.006756                 True
              600                0.640328              0.640410   -0.000082                False
              700                0.638786              0.649773   -0.010987                False
              800                0.640700              0.6572

## 06: Accomplished Goals 1-Way ANOVA Test


In [13]:
# Use df_da_full and df_va_full from cell 11 which includes "Accomplished Goals"
# (saved before they were overwritten in cells 17 and 19)
# These use FULL data (all agents, not filtered by consent count > 0)
groups_da = [
    df_da_full[df_da_full["monitoring_count"] == r]["Accomplished Goals"]
    for r in sorted(df_da_full["monitoring_count"].unique())
]

# One-way ANOVA for ConsentFirstAgent (DA) - Accomplished Goals
F_da, p_da = stats.f_oneway(*groups_da)

# Effect size (eta-squared) for DA
k_da = len(groups_da)                       # number of groups
ns_da = [len(g) for g in groups_da]
N_da = sum(ns_da)                          # total sample size
df_between_da = k_da - 1
df_within_da = N_da - k_da
eta_sq_da = (F_da * df_between_da) / (F_da * df_between_da + df_within_da)

print(f"ConsentFirstAgent (DA) - F: {F_da:.4f}, p: {p_da:.3e}, eta^2: {eta_sq_da:.4f}")

groups_va = [
    df_va_full[df_va_full["monitoring_count"] == r]["Accomplished Goals"]
    for r in sorted(df_va_full["monitoring_count"].unique())
]

# One-way ANOVA for MonitoringAgent (VA) - Accomplished Goals
F_va, p_va = stats.f_oneway(*groups_va)

# Effect size (eta-squared) for VA
k_va = len(groups_va)                       # number of groups
ns_va = [len(g) for g in groups_va]
N_va = sum(ns_va)                          # total sample size
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va = (F_va * df_between_va) / (F_va * df_between_va + df_within_va)

print(f"MonitoringAgent (VA) - F: {F_va:.4f}, p: {p_va:.3e}, eta^2: {eta_sq_va:.4f}")


ConsentFirstAgent (DA) - F: 2.0170, p: 4.622e-02, eta^2: 0.1678
MonitoringAgent (VA) - F: 0.8933, p: 5.346e-01, eta^2: 0.0820


## 08: Total Idle Time per Agent Normalized by Steps


In [14]:
# Use df_da_full and df_va_full from cell 18 which includes "tot_idle_time_normalized"
# (saved before they were overwritten in cells 19 and 20)
# These use FULL data (all agents, not filtered by consent count > 0)
groups_da = [
    df_da_full[df_da_full["monitoring_count"] == r]["tot_idle_time_normalized"]
    for r in sorted(df_da_full["monitoring_count"].unique())
]

# One-way ANOVA for ConsentFirstAgent (DA) - Total Idle Time Normalized
F_da, p_da = stats.f_oneway(*groups_da)

# Effect size (eta-squared) for DA
k_da = len(groups_da)                       # number of groups
ns_da = [len(g) for g in groups_da]
N_da = sum(ns_da)                          # total sample size
df_between_da = k_da - 1
df_within_da = N_da - k_da
eta_sq_da = (F_da * df_between_da) / (F_da * df_between_da + df_within_da)

# Get min / max values of the averages
max_tot_idle_time_normalized_da = df_da_full.groupby("monitoring_count")["tot_idle_time_normalized"].mean().max()
min_tot_idle_time_normalized_da = df_da_full.groupby("monitoring_count")["tot_idle_time_normalized"].mean().min()

print(f"ConsentFirstAgent (DA) - F: {F_da:.4f}, p: {p_da:.3e}, eta^2: {eta_sq_da:.4f}")
print(f"  Max: {max_tot_idle_time_normalized_da:.4f}, Min: {min_tot_idle_time_normalized_da:.4f}")

groups_va = [
    df_va_full[df_va_full["monitoring_count"] == r]["tot_idle_time_normalized"]
    for r in sorted(df_va_full["monitoring_count"].unique())
]

# One-way ANOVA for MonitoringAgent (VA) - Total Idle Time Normalized
F_va, p_va = stats.f_oneway(*groups_va)

# Effect size (eta-squared) for VA
k_va = len(groups_va)                       # number of groups
ns_va = [len(g) for g in groups_va]
N_va = sum(ns_va)                          # total sample size
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va = (F_va * df_between_va) / (F_va * df_between_va + df_within_va)

# Get min / max values of the averages
max_tot_idle_time_normalized_va = df_va_full.groupby("monitoring_count")["tot_idle_time_normalized"].mean().max()
min_tot_idle_time_normalized_va = df_va_full.groupby("monitoring_count")["tot_idle_time_normalized"].mean().min()

print(f"MonitoringAgent (VA) - F: {F_va:.4f}, p: {p_va:.3e}, eta^2: {eta_sq_va:.4f}")
print(f"  Max: {max_tot_idle_time_normalized_va:.4f}, Min: {min_tot_idle_time_normalized_va:.4f}")


ConsentFirstAgent (DA) - F: 0.7039, p: 7.038e-01, eta^2: 0.0658
  Max: 0.5411, Min: 0.5127
MonitoringAgent (VA) - F: 2.8870, p: 4.824e-03, eta^2: 0.2240
  Max: 0.5120, Min: 0.4688


## 08: Total Idle Time Normalized Mann-Whitney Test

In [15]:
from scipy.stats import mannwhitneyu

# ----------------------------------------------------------------------
# 1. Use df_da_full and df_va_full from cell 18 which includes "tot_idle_time_normalized"
#    These include ALL agents (not just those with consents)
# ----------------------------------------------------------------------

# ----------------------------------------------------------------------
# 2. Compare tot_idle_time_normalized between personas
# ----------------------------------------------------------------------

# Means per condition (align conditions across personas)
consent_first_means = df_da_full.groupby("monitoring_count")["tot_idle_time_normalized"].mean()
monitoring_means = df_va_full.groupby("monitoring_count")["tot_idle_time_normalized"].mean()

# Use union of all observed monitoring_count values and fill missing persona means with 0
all_counts = sorted(set(consent_first_means.index).union(set(monitoring_means.index)))
consent_first_aligned = consent_first_means.reindex(all_counts, fill_value=0)
monitoring_aligned = monitoring_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "monitoring_count": all_counts,
    "ConsentFirstAgent_mean": consent_first_aligned.values,
    "MonitoringAgent_mean": monitoring_aligned.values
})
comparison_df["Difference"] = (
    comparison_df["ConsentFirstAgent_mean"] -
    comparison_df["MonitoringAgent_mean"]
)
comparison_df["ConsentFirst_higher"] = comparison_df["Difference"] > 0

print("\nTotal Idle Time Normalized Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

# Count conditions
consent_first_higher_count = comparison_df["ConsentFirst_higher"].sum()
monitoring_higher_count = len(comparison_df) - consent_first_higher_count

print(f"\nConsentFirstAgent higher in {consent_first_higher_count}/"
      f"{len(comparison_df)} conditions")
print(f"MonitoringAgent higher in {monitoring_higher_count}/"
      f"{len(comparison_df)} conditions")

# ----------------------------------------------------------------------
# 3. Compare overall means
# ----------------------------------------------------------------------

overall_da = df_da_full["tot_idle_time_normalized"].mean()
overall_va = df_va_full["tot_idle_time_normalized"].mean()

print("\nOverall mean tot_idle_time_normalized:")
print(f"  ConsentFirstAgent: {overall_da:.4f}")
print(f"  MonitoringAgent:   {overall_va:.4f}")
print(f"  Difference:        {overall_da - overall_va:.4f}")

# ----------------------------------------------------------------------
# 4. Mann–Whitney U tests (non-parametric)
# ----------------------------------------------------------------------

print("\nMann–Whitney U Test (Total Idle Time Normalized):")
idle_da = df_da_full["tot_idle_time_normalized"]
idle_va = df_va_full["tot_idle_time_normalized"]

u_twosided, p_twosided = mannwhitneyu(idle_da, idle_va, alternative="two-sided")
u_da_greater, p_da_greater = mannwhitneyu(idle_da, idle_va, alternative="greater")
u_va_greater, p_va_greater = mannwhitneyu(idle_va, idle_da, alternative="greater")

print(f"  Two-sided:           p = {p_twosided:.3e}")
print(f"  DA > VA (one-sided): p = {p_da_greater:.3e}")
print(f"  VA > DA (one-sided): p = {p_va_greater:.3e}")

# Rank-biserial correlation (effect size) using two-sided U
n_da = len(idle_da)
n_va = len(idle_va)
if n_da > 0 and n_va > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_da * n_va)
    # Give direction based on overall means
    direction = 1 if overall_da > overall_va else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if overall_da > overall_va:
        print("  → SIGNIFICANT: ConsentFirstAgent has higher normalized idle time.")
    else:
        print("  → SIGNIFICANT: MonitoringAgent has higher normalized idle time.")
else:
    print("  → NO significant difference.")


Total Idle Time Normalized Comparison by Persona:
 monitoring_count  ConsentFirstAgent_mean  MonitoringAgent_mean  Difference  ConsentFirst_higher
                0                0.533912              0.000000    0.533912                 True
              100                0.521779              0.499995    0.021784                 True
              200                0.512651              0.487195    0.025455                 True
              300                0.528778              0.501620    0.027158                 True
              400                0.528086              0.497307    0.030778                 True
              500                0.535751              0.501201    0.034550                 True
              600                0.524390              0.485070    0.039320                 True
              700                0.523984              0.470095    0.053888                 True
              800                0.541071              0.480282    0.060789 

## 09: Finished Step 1-Way ANOVA Tests

In [16]:
# 09: Finished Step 1-Way ANOVA Tests

# Build run-level averages for Finished Step (all agents)
df_finished = final_agent_values.copy()

run_level_df_finished = (
    df_finished.groupby(["monitoring_count", "seed", "Agent Persona"])[["Finished Step"]]
    .mean()
    .reset_index()
)

df_da_finished = run_level_df_finished[run_level_df_finished["Agent Persona"] == "ConsentFirstAgent"]
df_va_finished = run_level_df_finished[run_level_df_finished["Agent Persona"] == "MonitoringAgent"]

# 1-way ANOVA for ConsentFirstAgent
print("ConsentFirstAgent - Finished Step ANOVA:")
groups_da = [
    df_da_finished[df_da_finished["monitoring_count"] == r]["Finished Step"]
    for r in sorted(df_da_finished["monitoring_count"].unique())
]
F_da, p_da = stats.f_oneway(*groups_da)

# Effect size (eta-squared) for DA
k_da = len(groups_da)                       # number of groups
ns_da = [len(g) for g in groups_da]
N_da = sum(ns_da)                          # total sample size
df_between_da = k_da - 1
df_within_da = N_da - k_da
eta_sq_da = (F_da * df_between_da) / (F_da * df_between_da + df_within_da)

# Get min / max values of the averages
max_finished_step_da = df_da_finished.groupby("monitoring_count")["Finished Step"].mean().max()
min_finished_step_da = df_da_finished.groupby("monitoring_count")["Finished Step"].mean().min()

print(f"F: {F_da:.4f}, p: {p_da:.3e}, eta^2: {eta_sq_da:.4f}")
print(f"  Max: {max_finished_step_da:.4f}, Min: {min_finished_step_da:.4f}")

# 1-way ANOVA for MonitoringAgent
print("\nMonitoringAgent - Finished Step ANOVA:")
groups_va = [
    df_va_finished[df_va_finished["monitoring_count"] == r]["Finished Step"]
    for r in sorted(df_va_finished["monitoring_count"].unique())
]
F_va, p_va = stats.f_oneway(*groups_va)

# Effect size (eta-squared) for VA
k_va = len(groups_va)                       # number of groups
ns_va = [len(g) for g in groups_va]
N_va = sum(ns_va)                          # total sample size
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va = (F_va * df_between_va) / (F_va * df_between_va + df_within_va)

# Get min / max values of the averages
max_finished_step_va = df_va_finished.groupby("monitoring_count")["Finished Step"].mean().max()
min_finished_step_va = df_va_finished.groupby("monitoring_count")["Finished Step"].mean().min()

print(f"F: {F_va:.4f}, p: {p_va:.3e}, eta^2: {eta_sq_va:.4f}")
print(f"  Max: {max_finished_step_va:.4f}, Min: {min_finished_step_va:.4f}")



ConsentFirstAgent - Finished Step ANOVA:
F: 39.7233, p: 1.307e-27, eta^2: 0.7989
  Max: 21.7626, Min: 14.9730

MonitoringAgent - Finished Step ANOVA:
F: 53.1576, p: 3.308e-32, eta^2: 0.8417
  Max: 18.6520, Min: 13.1252


# 09: Finished Step Mann–Whitney U Test

In [17]:
# 09: Finished Step Mann–Whitney U Test

from scipy.stats import mannwhitneyu
import pandas as pd

# Use the run_level_df_finished, df_da_finished, and df_va_finished
# created in the previous cell.

# ----------------------------------------------------------------------
# 1. Compare Finished Step between personas per condition
# ----------------------------------------------------------------------

# Means per condition (align conditions across personas)
consent_first_means = df_da_finished.groupby("monitoring_count")["Finished Step"].mean()
monitoring_means = df_va_finished.groupby("monitoring_count")["Finished Step"].mean()

# Use union of all observed monitoring_count values and fill missing persona means with 0
all_counts = sorted(set(consent_first_means.index).union(set(monitoring_means.index)))
consent_first_aligned = consent_first_means.reindex(all_counts, fill_value=0)
monitoring_aligned = monitoring_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "monitoring_count": all_counts,
    "ConsentFirstAgent_mean": consent_first_aligned.values,
    "MonitoringAgent_mean": monitoring_aligned.values
})
comparison_df["Difference"] = (
    comparison_df["ConsentFirstAgent_mean"] -
    comparison_df["MonitoringAgent_mean"]
)
comparison_df["ConsentFirst_higher"] = comparison_df["Difference"] > 0

print("\nFinished Step Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

# Count conditions
consent_first_higher_count = comparison_df["ConsentFirst_higher"].sum()
monitoring_higher_count = len(comparison_df) - consent_first_higher_count

print(f"\nConsentFirstAgent higher in {consent_first_higher_count}/"
      f"{len(comparison_df)} conditions")
print(f"MonitoringAgent higher in {monitoring_higher_count}/"
      f"{len(comparison_df)} conditions")

# ----------------------------------------------------------------------
# 2. Compare overall means
# ----------------------------------------------------------------------

print("\nMann–Whitney U Test (Finished Step):")

finished_da = df_da_finished["Finished Step"]
finished_va = df_va_finished["Finished Step"]

u_twosided, p_twosided = mannwhitneyu(finished_da, finished_va, alternative="two-sided")
u_da_greater, p_da_greater = mannwhitneyu(finished_da, finished_va, alternative="greater")
u_va_greater, p_va_greater = mannwhitneyu(finished_va, finished_da, alternative="greater")

print(f"  Two-sided:           p = {p_twosided:.3e}")
print(f"  DA > VA (one-sided): p = {p_da_greater:.3e}")
print(f"  VA > DA (one-sided): p = {p_va_greater:.3e}")

overall_da = finished_da.mean()
overall_va = finished_va.mean()
print(f"  ConsentFirstAgent mean finished step: {overall_da:.2f}")
print(f"  MonitoringAgent   mean finished step: {overall_va:.2f}")
print(f"  Difference:         {overall_da - overall_va:.2f}")

# Rank-biserial correlation (effect size) using two-sided U
n_da = len(finished_da)
n_va = len(finished_va)
if n_da > 0 and n_va > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_da * n_va)
    # Give direction based on overall means
    direction = 1 if overall_da > overall_va else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if overall_da > overall_va:
        print("  → SIGNIFICANT: ConsentFirstAgent finishes later on average.")
    else:
        print("  → SIGNIFICANT: MonitoringAgent finishes later on average.")
else:
    print("  → NO significant difference in finished step.")




Finished Step Comparison by Persona:
 monitoring_count  ConsentFirstAgent_mean  MonitoringAgent_mean  Difference  ConsentFirst_higher
                0               21.762614              0.000000   21.762614                 True
              100               19.335556             18.652000    0.683556                 True
              200               17.920612             17.181912    0.738700                 True
              300               17.104357             16.383667    0.720690                 True
              400               16.616420             15.825856    0.790564                 True
              500               16.067000             15.225600    0.841400                 True
              600               15.722500             14.767000    0.955500                 True
              700               15.522333             14.236429    1.285905                 True
              800               15.258000             13.887000    1.371000              

# Cumulative Graphs, Wilcoxon Test:

Even if the difference of Accomplished Goals for DAs and VAs might be small, it can be consistent.
Wilcoxon test shows this.


In [18]:
df = all_agent_values_df_all_steps[["seed", "agent_config", "monitoring_count", "Step",  "AgentID", "Agent Persona", "Accomplished Goals"]]

results = []

for r in sorted(df["monitoring_count"].unique()):

    df_r = df[df["monitoring_count"] == r]

    # Compute per-seed average accomplished goals per agent
    da_seed_means = (
        df_r[df_r["Agent Persona"] == "ConsentFirstAgent"]
        .groupby("seed")["Accomplished Goals"]
        .mean()
    )

    va_seed_means = (
        df_r[df_r["Agent Persona"] == "MonitoringAgent"]
        .groupby("seed")["Accomplished Goals"]
        .mean()
    )

    # Ensure paired samples (same seeds)
    common_seeds = da_seed_means.index.intersection(va_seed_means.index)

    da_vals = da_seed_means.loc[common_seeds]
    va_vals = va_seed_means.loc[common_seeds]

    # Wilcoxon signed-rank test
    stat, p = wilcoxon(va_vals, da_vals)

    # Store results
    results.append({
        "monitoring_count": r,
        "wilcoxon_stat": stat,
        "p_value": p,
        "VA_mean": va_vals.mean(),
        "DA_mean": da_vals.mean(),
    })

results_df = pd.DataFrame(results)
print(results_df)


/Users/efeonal/py_envs/MESA_thesis/venv/lib/python3.12/site-packages/scipy/_lib/_util.py:999: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return fun(*args, **kwargs)


    monitoring_count  wilcoxon_stat   p_value   VA_mean   DA_mean
0                  0            NaN       NaN       NaN       NaN
1                100            3.0  0.009766  1.624080  1.571249
2                200            2.0  0.005859  1.641737  1.577276
3                300            1.0  0.003906  1.588034  1.524360
4                400            0.0  0.001953  1.595625  1.520022
5                500            0.0  0.001953  1.578325  1.489369
6                600            0.0  0.001953  1.618515  1.511297
7                700            0.0  0.001953  1.653724  1.505434
8                800            0.0  0.001953  1.618680  1.449360
9                900            0.0  0.001953  1.639336  1.441074
10              1000            NaN       NaN       NaN       NaN


/Users/efeonal/py_envs/MESA_thesis/venv/lib/python3.12/site-packages/scipy/_lib/_util.py:999: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return fun(*args, **kwargs)


## 10: Average Distinct Agents Interacted as R, 1-Way ANOVA Tests


In [19]:
# Create run-level dataframe for "Number of Distinct Agents Interacted as R"
# This includes ALL agents (not just those with consents)
df_distinct_r = final_agent_values.copy()

run_level_df_distinct_r = (
    df_distinct_r.groupby(["monitoring_count", "seed", "Agent Persona"])[
        ["Number of Distinct Agents Interacted as R"]
    ]
    .mean()
    .reset_index()
)

df_da_distinct_r = run_level_df_distinct_r[run_level_df_distinct_r["Agent Persona"] == "ConsentFirstAgent"]
df_va_distinct_r = run_level_df_distinct_r[run_level_df_distinct_r["Agent Persona"] == "MonitoringAgent"]

# 1-way ANOVA for ConsentFirstAgent
groups_da = [
    df_da_distinct_r[df_da_distinct_r["monitoring_count"] == r]["Number of Distinct Agents Interacted as R"]
    for r in sorted(df_da_distinct_r["monitoring_count"].unique())
]

F_da, p_da = stats.f_oneway(*groups_da)

# Effect size (eta-squared) for DA
k_da = len(groups_da)                       # number of groups
ns_da = [len(g) for g in groups_da]
N_da = sum(ns_da)                          # total sample size
df_between_da = k_da - 1
df_within_da = N_da - k_da
eta_sq_da = (F_da * df_between_da) / (F_da * df_between_da + df_within_da)

# Get min / max values of the averages
max_distinct_agents_r_da = df_da_distinct_r.groupby("monitoring_count")["Number of Distinct Agents Interacted as R"].mean().max()
min_distinct_agents_r_da = df_da_distinct_r.groupby("monitoring_count")["Number of Distinct Agents Interacted as R"].mean().min()

print(f"ConsentFirstAgent - F: {F_da:.4f}, p: {p_da:.3e}, eta^2: {eta_sq_da:.4f}")
print(f"  Max: {max_distinct_agents_r_da:.4f}, Min: {min_distinct_agents_r_da:.4f}")

# 1-way ANOVA for MonitoringAgent
groups_va = [
    df_va_distinct_r[df_va_distinct_r["monitoring_count"] == r]["Number of Distinct Agents Interacted as R"]
    for r in sorted(df_va_distinct_r["monitoring_count"].unique())
]

F_va, p_va = stats.f_oneway(*groups_va)

# Effect size (eta-squared) for VA
k_va = len(groups_va)                       # number of groups
ns_va = [len(g) for g in groups_va]
N_va = sum(ns_va)                          # total sample size
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va = (F_va * df_between_va) / (F_va * df_between_va + df_within_va)

# Get min / max values of the averages
max_distinct_agents_r_va = df_va_distinct_r.groupby("monitoring_count")["Number of Distinct Agents Interacted as R"].mean().max()
min_distinct_agents_r_va = df_va_distinct_r.groupby("monitoring_count")["Number of Distinct Agents Interacted as R"].mean().min()

print(f"MonitoringAgent - F: {F_va:.4f}, p: {p_va:.3e}, eta^2: {eta_sq_va:.4f}")
print(f"  Max: {max_distinct_agents_r_va:.4f}, Min: {min_distinct_agents_r_va:.4f}")


ConsentFirstAgent - F: 5.9986, p: 1.500e-06, eta^2: 0.3749
  Max: 9.7026, Min: 9.2360
MonitoringAgent - F: 1.7401, p: 9.129e-02, eta^2: 0.1482
  Max: 10.4505, Min: 10.1218


## 10: Average Distinct Agents Interacted as R, Mann-Whitney Test


In [20]:
# ----------------------------------------------------------------------
# 1. Use df_da_distinct_r and df_va_distinct_r from cell 29 (ANOVA test)
#    These include ALL agents (not just those with consents)
# ----------------------------------------------------------------------

# ----------------------------------------------------------------------
# 2. Compare "Number of Distinct Agents Interacted as R" between personas
# ----------------------------------------------------------------------

# Means per condition (align conditions across personas)
consent_first_means = df_da_distinct_r.groupby("monitoring_count")["Number of Distinct Agents Interacted as R"].mean()
monitoring_means = df_va_distinct_r.groupby("monitoring_count")["Number of Distinct Agents Interacted as R"].mean()

# Use union of all observed monitoring_count values and fill missing persona means with 0
all_counts = sorted(set(consent_first_means.index).union(set(monitoring_means.index)))
consent_first_aligned = consent_first_means.reindex(all_counts, fill_value=0)
monitoring_aligned = monitoring_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "monitoring_count": all_counts,
    "ConsentFirstAgent_mean": consent_first_aligned.values,
    "MonitoringAgent_mean": monitoring_aligned.values
})
comparison_df["Difference"] = (
    comparison_df["ConsentFirstAgent_mean"] -
    comparison_df["MonitoringAgent_mean"]
)
comparison_df["ConsentFirst_higher"] = comparison_df["Difference"] > 0

print("\nDistinct Agents Interacted as R Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

# Count conditions
consent_first_higher_count = comparison_df["ConsentFirst_higher"].sum()
monitoring_higher_count = len(comparison_df) - consent_first_higher_count

print(f"\nConsentFirstAgent higher in {consent_first_higher_count}/"
      f"{len(comparison_df)} conditions")
print(f"MonitoringAgent higher in {monitoring_higher_count}/"
      f"{len(comparison_df)} conditions")

# ----------------------------------------------------------------------
# 3. Compare overall means
# ----------------------------------------------------------------------

overall_da = df_da_distinct_r["Number of Distinct Agents Interacted as R"].mean()
overall_va = df_va_distinct_r["Number of Distinct Agents Interacted as R"].mean()

print("\nOverall mean Distinct Agents Interacted as R:")
print(f"  ConsentFirstAgent: {overall_da:.4f}")
print(f"  MonitoringAgent:   {overall_va:.4f}")
print(f"  Difference:         {overall_da - overall_va:.4f}")

# ----------------------------------------------------------------------
# 4. Mann–Whitney U tests (non-parametric)
# ----------------------------------------------------------------------

from scipy.stats import mannwhitneyu

print("\nMann–Whitney U Test (Distinct Agents Interacted as R):")
distinct_da = df_da_distinct_r["Number of Distinct Agents Interacted as R"]
distinct_va = df_va_distinct_r["Number of Distinct Agents Interacted as R"]

u_twosided, p_twosided = mannwhitneyu(distinct_da, distinct_va, alternative="two-sided")
u_da_greater, p_da_greater = mannwhitneyu(distinct_da, distinct_va, alternative="greater")
u_va_greater, p_va_greater = mannwhitneyu(distinct_va, distinct_da, alternative="greater")

print(f"  Two-sided:           p = {p_twosided:.3e}")
print(f"  DA > VA (one-sided): p = {p_da_greater:.3e}")
print(f"  VA > DA (one-sided): p = {p_va_greater:.3e}")

# Rank-biserial correlation (effect size) using two-sided U
n_da = len(distinct_da)
n_va = len(distinct_va)
if n_da > 0 and n_va > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_da * n_va)
    # Give direction based on overall means
    direction = 1 if overall_da > overall_va else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if overall_da > overall_va:
        print("  → SIGNIFICANT: ConsentFirstAgent has higher distinct agents interacted as R.")
    else:
        print("  → SIGNIFICANT: MonitoringAgent has higher distinct agents interacted as R.")
else:
    print("  → NO significant difference.")



Distinct Agents Interacted as R Comparison by Persona:
 monitoring_count  ConsentFirstAgent_mean  MonitoringAgent_mean  Difference  ConsentFirst_higher
                0                9.702600              0.000000    9.702600                 True
              100                9.593000             10.415000   -0.822000                False
              200                9.539125             10.450500   -0.911375                False
              300                9.510429             10.343333   -0.832905                False
              400                9.501167             10.424000   -0.922833                False
              500                9.503400             10.379200   -0.875800                False
              600                9.452750             10.369833   -0.917083                False
              700                9.454000             10.314857   -0.860857                False
              800                9.367500             10.297375   -0.92

In [21]:
# 09: Resource Conflicts per Agent  — 1-way ANOVA

from scipy import stats
import numpy as np

# Start from final_agent_values (one row per agent at final step per run)
df_rc = final_agent_values.copy()

# Normalize by steps
df_rc["resource_conflicts_norm"] = df_rc["Resource Conflicts"] #/ df_rc["Step"]

# Split by persona
df_da_rc = df_rc[df_rc["Agent Persona"] == "ConsentFirstAgent"]
df_va_rc = df_rc[df_rc["Agent Persona"] == "MonitoringAgent"]

# ---- ConsentFirstAgent (DA) ----
groups_da_rc = [
    df_da_rc[df_da_rc["monitoring_count"] == r]["resource_conflicts_norm"].values
    for r in sorted(df_da_rc["monitoring_count"].unique())
]

F_da_rc, p_da_rc = stats.f_oneway(*groups_da_rc)

k_da = len(groups_da_rc)
ns_da = [len(g) for g in groups_da_rc]
N_da = sum(ns_da)
df_between_da = k_da - 1
df_within_da = N_da - k_da
eta_sq_da_rc = (F_da_rc * df_between_da) / (F_da_rc * df_between_da + df_within_da)

max_rc_da = df_da_rc.groupby("monitoring_count")["resource_conflicts_norm"].mean().max()
min_rc_da = df_da_rc.groupby("monitoring_count")["resource_conflicts_norm"].mean().min()

print("Resource Conflicts — ConsentFirstAgent (DA)")
print(f"  F: {F_da_rc:.4f}, p: {p_da_rc:.3e}, eta^2: {eta_sq_da_rc:.4f}")
print(f"  Max mean: {max_rc_da:.4f}, Min mean: {min_rc_da:.4f}\n")

# ---- MonitoringAgent (VA) ----
groups_va_rc = [
    df_va_rc[df_va_rc["monitoring_count"] == r]["resource_conflicts_norm"].values
    for r in sorted(df_va_rc["monitoring_count"].unique())
]

F_va_rc, p_va_rc = stats.f_oneway(*groups_va_rc)

k_va = len(groups_va_rc)
ns_va = [len(g) for g in groups_va_rc]
N_va = sum(ns_va)
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va_rc = (F_va_rc * df_between_va) / (F_va_rc * df_between_va + df_within_va)

max_rc_va = df_va_rc.groupby("monitoring_count")["resource_conflicts_norm"].mean().max()
min_rc_va = df_va_rc.groupby("monitoring_count")["resource_conflicts_norm"].mean().min()

print("Resource Conflicts — MonitoringAgent (VA)")
print(f"  F: {F_va_rc:.4f}, p: {p_va_rc:.3e}, eta^2: {eta_sq_va_rc:.4f}")
print(f"  Max mean: {max_rc_va:.4f}, Min mean: {min_rc_va:.4f}")



Resource Conflicts — ConsentFirstAgent (DA)
  F: 281.0863, p: 0.000e+00, eta^2: 0.0225
  Max mean: 10.0093, Min mean: 6.3050

Resource Conflicts — MonitoringAgent (VA)
  F: 1556.3223, p: 0.000e+00, eta^2: 0.1130
  Max mean: 4.5600, Min mean: 0.8385


In [22]:
# 11: Resource Conflicts Mann–Whitney U Test (DA vs VA)

from scipy.stats import mannwhitneyu
import pandas as pd

# ----------------------------------------------------------------------
# 1. Prepare data properly: aggregate per run (like df_da_full pattern)
# ----------------------------------------------------------------------

df_all_agents = final_agent_values.copy()

# Get simulation's total steps per run
sim_steps_per_run = (
    df_all_agents.groupby(["monitoring_count", "seed"])["Step"]
    .max()
    .reset_index()
    .rename(columns={"Step": "sim_total_steps"})
)

# Aggregate to run-level: average Resource Conflicts per agent type per run
run_level_df = (
    df_all_agents.groupby(["monitoring_count", "seed", "Agent Persona"])[["Resource Conflicts"]]
    .mean()
    .reset_index()
)

# Merge simulation steps
run_level_df = run_level_df.merge(sim_steps_per_run, on=["monitoring_count", "seed"])

# Normalize by simulation's total steps (matching graph: avg_resource_conflicts / avg_steps_overall)
run_level_df["resource_conflicts_norm"] = (
    run_level_df["Resource Conflicts"] #/ run_level_df["sim_total_steps"]
)

# Split by persona
df_da = run_level_df[run_level_df["Agent Persona"] == "ConsentFirstAgent"]
df_va = run_level_df[run_level_df["Agent Persona"] == "MonitoringAgent"]

# Per-condition means (for a quick descriptive table)
da_means = df_da.groupby("monitoring_count")["resource_conflicts_norm"].mean()
va_means = df_va.groupby("monitoring_count")["resource_conflicts_norm"].mean()

all_counts = sorted(set(da_means.index).union(set(va_means.index)))
da_aligned = da_means.reindex(all_counts, fill_value=0)
va_aligned = va_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "monitoring_count": all_counts,
    "ConsentFirstAgent_mean": da_aligned.values,
    "MonitoringAgent_mean": va_aligned.values,
})
comparison_df["Difference"] = comparison_df["ConsentFirstAgent_mean"] - comparison_df["MonitoringAgent_mean"]
comparison_df["ConsentFirst_higher"] = comparison_df["Difference"] > 0

print("\nResource Conflicts Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

consent_first_higher_count = comparison_df["ConsentFirst_higher"].sum()
monitoring_higher_count = len(comparison_df) - consent_first_higher_count
print(f"\nConsentFirstAgent higher in {consent_first_higher_count}/{len(comparison_df)} conditions")
print(f"MonitoringAgent higher in {monitoring_higher_count}/{len(comparison_df)} conditions")

# Global Mann–Whitney U test on run-level values (all seeds × configs)
print("\nMann–Whitney U Test (Resource Conflicts):")
cons_da = df_da["resource_conflicts_norm"]
cons_va = df_va["resource_conflicts_norm"]

u_twosided, p_twosided = mannwhitneyu(cons_da, cons_va, alternative="two-sided")
u_da_greater, p_da_greater = mannwhitneyu(cons_da, cons_va, alternative="greater")
u_va_greater, p_va_greater = mannwhitneyu(cons_va, cons_da, alternative="greater")

print(f"  Two-sided:           p = {p_twosided:.3e}")
print(f"  DA > VA (one-sided): p = {p_da_greater:.3e}")
print(f"  VA > DA (one-sided): p = {p_va_greater:.3e}")

mean_da = cons_da.mean()
mean_va = cons_va.mean()
print(f"  ConsentFirstAgent mean RC: {mean_da:.4f}")
print(f"  MonitoringAgent    mean RC: {mean_va:.4f}")
print(f"  Difference:                     {mean_da - mean_va:.4f}")

n_da = len(cons_da)
n_va = len(cons_va)
if n_da > 0 and n_va > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_da * n_va)
    direction = 1 if mean_da > mean_va else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if mean_da > mean_va:
        print("  → SIGNIFICANT: ConsentFirstAgent has higher RC.")
    else:
        print("  → SIGNIFICANT: MonitoringAgent has higher RC.")
else:
    print("  → NO significant difference in RC.")




Resource Conflicts Comparison by Persona:
 monitoring_count  ConsentFirstAgent_mean  MonitoringAgent_mean  Difference  ConsentFirst_higher
                0               10.009300              0.000000   10.009300                 True
              100                8.624556              4.560000    4.064556                 True
              200                7.723875              4.234000    3.489875                 True
              300                7.259286              3.745000    3.514286                 True
              400                6.995500              3.460250    3.535250                 True
              500                6.729400              2.972600    3.756800                 True
              600                6.544250              2.653500    3.890750                 True
              700                6.496000              2.157429    4.338571                 True
              800                6.508000              1.763625    4.744375         

In [23]:
# 12: Counter Conflict Goals/Step Mann–Whitney U Test (DA vs VA)

from scipy.stats import mannwhitneyu
import pandas as pd

# ----------------------------------------------------------------------
# 1. Prepare data properly: aggregate per run (like df_da_full pattern)
# ----------------------------------------------------------------------

df_all_agents = final_agent_values.copy()

# Get simulation's total steps per run
sim_steps_per_run = (
    df_all_agents.groupby(["monitoring_count", "seed"])["Step"]
    .max()
    .reset_index()
    .rename(columns={"Step": "sim_total_steps"})
)

# Aggregate to run-level: average Counter Goal Accomplishments per agent type per run
run_level_df = (
    df_all_agents.groupby(["monitoring_count", "seed", "Agent Persona"])[["Counter Conflict Goal Accomplishments"]]
    .mean()
    .reset_index()
)

# Merge simulation steps
run_level_df = run_level_df.merge(sim_steps_per_run, on=["monitoring_count", "seed"])

# Normalize by simulation's total steps (matching graph: avg_counter_goals / avg_steps_overall)
run_level_df["counter_goals_norm"] = (
    run_level_df["Counter Conflict Goal Accomplishments"] / run_level_df["sim_total_steps"]
)

# Split by persona
df_da = run_level_df[run_level_df["Agent Persona"] == "ConsentFirstAgent"]
df_va = run_level_df[run_level_df["Agent Persona"] == "MonitoringAgent"]

# Per-condition means (for a quick descriptive table)
da_means = df_da.groupby("monitoring_count")["counter_goals_norm"].mean()
va_means = df_va.groupby("monitoring_count")["counter_goals_norm"].mean()

all_counts = sorted(set(da_means.index).union(set(va_means.index)))
da_aligned = da_means.reindex(all_counts, fill_value=0)
va_aligned = va_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "monitoring_count": all_counts,
    "ConsentFirstAgent_mean": da_aligned.values,
    "MonitoringAgent_mean": va_aligned.values,
})
comparison_df["Difference"] = comparison_df["ConsentFirstAgent_mean"] - comparison_df["MonitoringAgent_mean"]
comparison_df["ConsentFirst_higher"] = comparison_df["Difference"] > 0

print("\nCounter Goals/Step Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

consent_first_higher_count = comparison_df["ConsentFirst_higher"].sum()
monitoring_higher_count = len(comparison_df) - consent_first_higher_count
print(f"\nConsentFirstAgent higher in {consent_first_higher_count}/{len(comparison_df)} conditions")
print(f"MonitoringAgent higher in {monitoring_higher_count}/{len(comparison_df)} conditions")

# Global Mann–Whitney U test on run-level values (all seeds × configs)
print("\nMann–Whitney U Test (Counter Conflict Goals/Step):")
cons_da = df_da["counter_goals_norm"]
cons_va = df_va["counter_goals_norm"]

u_twosided, p_twosided = mannwhitneyu(cons_da, cons_va, alternative="two-sided")
u_da_greater, p_da_greater = mannwhitneyu(cons_da, cons_va, alternative="greater")
u_va_greater, p_va_greater = mannwhitneyu(cons_va, cons_da, alternative="greater")

print(f"  Two-sided:           p = {p_twosided:.3e}")
print(f"  DA > VA (one-sided): p = {p_da_greater:.3e}")
print(f"  VA > DA (one-sided): p = {p_va_greater:.3e}")

mean_da = cons_da.mean()
mean_va = cons_va.mean()
print(f"  ConsentFirstAgent mean counter-goals/step: {mean_da:.4f}")
print(f"  MonitoringAgent    mean counter-goals/step: {mean_va:.4f}")
print(f"  Difference:                                 {mean_da - mean_va:.4f}")

n_da = len(cons_da)
n_va = len(cons_va)
if n_da > 0 and n_va > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_da * n_va)
    direction = 1 if mean_da > mean_va else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if mean_da > mean_va:
        print("  → SIGNIFICANT: ConsentFirstAgent has higher counter-goals/step.")
    else:
        print("  → SIGNIFICANT: MonitoringAgent has higher counter-goals/step.")
else:
    print("  → NO significant difference in counter-goals/step.")




Counter Goals/Step Comparison by Persona:
 monitoring_count  ConsentFirstAgent_mean  MonitoringAgent_mean  Difference  ConsentFirst_higher
                0                0.025908              0.000000    0.025908                 True
              100                0.030260              0.025163    0.005097                 True
              200                0.032664              0.029178    0.003487                 True
              300                0.035380              0.030264    0.005116                 True
              400                0.036142              0.031908    0.004234                 True
              500                0.038638              0.030814    0.007824                 True
              600                0.038589              0.029303    0.009286                 True
              700                0.038667              0.026393    0.012273                 True
              800                0.041020              0.022676    0.018344         

In [24]:
# 10: Counter Goal Accomplishments per Agent Normalized by Steps — 1-way ANOVA

# Start from final_agent_values (one row per agent at final step per run)
df_cg = final_agent_values.copy()

# Normalize by steps
df_cg["counter_goals_norm"] = (
    df_cg["Counter Conflict Goal Accomplishments"] / df_cg["Step"]
)

# Split by persona
df_da_cg = df_cg[df_cg["Agent Persona"] == "ConsentFirstAgent"]
df_va_cg = df_cg[df_cg["Agent Persona"] == "MonitoringAgent"]

# ---- ConsentFirstAgent (DA) ----
groups_da_cg = [
    df_da_cg[df_da_cg["monitoring_count"] == r]["counter_goals_norm"].values
    for r in sorted(df_da_cg["monitoring_count"].unique())
]

F_da_cg, p_da_cg = stats.f_oneway(*groups_da_cg)

k_da = len(groups_da_cg)
ns_da = [len(g) for g in groups_da_cg]
N_da = sum(ns_da)
df_between_da = k_da - 1
df_within_da = N_da - k_da
eta_sq_da_cg = (F_da_cg * df_between_da) / (F_da_cg * df_between_da + df_within_da)

max_cg_da = df_da_cg.groupby("monitoring_count")["counter_goals_norm"].mean().max()
min_cg_da = df_da_cg.groupby("monitoring_count")["counter_goals_norm"].mean().min()

print("Counter Goals/Step — ConsentFirstAgent (DA)")
print(f"  F: {F_da_cg:.4f}, p: {p_da_cg:.3e}, eta^2: {eta_sq_da_cg:.4f}")
print(f"  Max mean: {max_cg_da:.4f}, Min mean: {min_cg_da:.4f}\n")

# ---- MonitoringAgent (VA) ----
groups_va_cg = [
    df_va_cg[df_va_cg["monitoring_count"] == r]["counter_goals_norm"].values
    for r in sorted(df_va_cg["monitoring_count"].unique())
]

F_va_cg, p_va_cg = stats.f_oneway(*groups_va_cg)

k_va = len(groups_va_cg)
ns_va = [len(g) for g in groups_va_cg]
N_va = sum(ns_va)
df_between_va = k_va - 1
df_within_va = N_va - k_va
eta_sq_va_cg = (F_va_cg * df_between_va) / (F_va_cg * df_between_va + df_within_va)

max_cg_va = df_va_cg.groupby("monitoring_count")["counter_goals_norm"].mean().max()
min_cg_va = df_va_cg.groupby("monitoring_count")["counter_goals_norm"].mean().min()

print("Counter Goals/Step — MonitoringAgent (VA)")
print(f"  F: {F_va_cg:.4f}, p: {p_va_cg:.3e}, eta^2: {eta_sq_va_cg:.4f}")
print(f"  Max mean: {max_cg_va:.4f}, Min mean: {min_cg_va:.4f}")



Counter Goals/Step — ConsentFirstAgent (DA)
  F: 130.0959, p: 5.025e-245, eta^2: 0.0105
  Max mean: 0.0419, Min mean: 0.0259

Counter Goals/Step — MonitoringAgent (VA)
  F: 266.7145, p: 0.000e+00, eta^2: 0.0214
  Max mean: 0.0319, Min mean: 0.0145


## 11: Counter Goal Accomplishments per Resource Conflict — 1-way ANOVA


In [25]:
# 11: Counter Goal Accomplishments per Resource Conflict — 1-way ANOVA

from scipy import stats
import numpy as np

# Start from final_agent_values (one row per agent at final step per run)
df_cg_rc = final_agent_values.copy()

# Calculate the ratio: Counter Goal Accomplishments / Resource Conflicts
# Handle division by zero by replacing 0 with NaN
df_cg_rc["counter_goal_per_rc"] = (
    df_cg_rc["Counter Conflict Goal Accomplishments"] / 
    df_cg_rc["Resource Conflicts"].replace(0, np.nan)
)

# Aggregate per run: sum counter goals and resource conflicts per persona per run, then divide
# This matches the calculation method in the analysis scripts
run_level_cg_rc = (
    df_cg_rc.groupby(["agent_config", "seed", "Agent Persona"])
    .agg({
        "Counter Conflict Goal Accomplishments": "sum",
        "Resource Conflicts": "sum"
    })
    .reset_index()
)

# Calculate ratio at run level (sum first, then divide)
run_level_cg_rc["counter_goal_per_rc"] = (
    run_level_cg_rc["Counter Conflict Goal Accomplishments"] / 
    run_level_cg_rc["Resource Conflicts"].replace(0, np.nan)
)

# Split by persona
df_da_cg_rc = run_level_cg_rc[run_level_cg_rc["Agent Persona"] == "ConsentFirstAgent"]
df_va_cg_rc = run_level_cg_rc[run_level_cg_rc["Agent Persona"] == "MonitoringAgent"]

# Extract consent_first_count from agent_config for grouping
# agent_config format: "ConsentFirst-FiftyFifty-Monitoring" (e.g., "100-0-900")
# So consent_first_count is the first part (index 0), monitoring_count is the last part (index -1)
if len(df_da_cg_rc) > 0:
    sample_config = df_da_cg_rc['agent_config'].iloc[0]
    parts = str(sample_config).split("-")
    if len(parts) >= 3:
        df_da_cg_rc["consent_first_count"] = df_da_cg_rc["agent_config"].str.split("-").str[0].astype(int)
    else:
        df_da_cg_rc["consent_first_count"] = 0

if len(df_va_cg_rc) > 0:
    sample_config = df_va_cg_rc['agent_config'].iloc[0]
    parts = str(sample_config).split("-")
    if len(parts) >= 3:
        df_va_cg_rc["consent_first_count"] = df_va_cg_rc["agent_config"].str.split("-").str[0].astype(int)
    else:
        df_va_cg_rc["consent_first_count"] = 0

# ---- ConsentFirstAgent (DA) ----
groups_da_cg_rc = [
    df_da_cg_rc[df_da_cg_rc["consent_first_count"] == r]["counter_goal_per_rc"].dropna().values
    for r in sorted(df_da_cg_rc["consent_first_count"].unique())
]

# Filter out empty groups
groups_da_cg_rc = [g for g in groups_da_cg_rc if len(g) > 0]

if len(groups_da_cg_rc) > 0:
    F_da_cg_rc, p_da_cg_rc = stats.f_oneway(*groups_da_cg_rc)
    
    k_da = len(groups_da_cg_rc)
    ns_da = [len(g) for g in groups_da_cg_rc]
    N_da = sum(ns_da)
    df_between_da = k_da - 1
    df_within_da = N_da - k_da
    eta_sq_da_cg_rc = (F_da_cg_rc * df_between_da) / (F_da_cg_rc * df_between_da + df_within_da) if (F_da_cg_rc * df_between_da + df_within_da) > 0 else 0
    
    max_cg_rc_da = df_da_cg_rc.groupby("consent_first_count")["counter_goal_per_rc"].mean().max()
    min_cg_rc_da = df_da_cg_rc.groupby("consent_first_count")["counter_goal_per_rc"].mean().min()
    
    print("Counter Goals/Resource Conflicts — ConsentFirstAgent (DA)")
    print(f"  F: {F_da_cg_rc:.4f}, p: {p_da_cg_rc:.3e}, eta^2: {eta_sq_da_cg_rc:.4f}")
    print(f"  Max mean: {max_cg_rc_da:.4f}, Min mean: {min_cg_rc_da:.4f}\n")
else:
    print("Counter Goals/Resource Conflicts — ConsentFirstAgent (DA)")
    print("  No valid data points\n")

# ---- MonitoringAgent (VA) ----
groups_va_cg_rc = [
    df_va_cg_rc[df_va_cg_rc["consent_first_count"] == r]["counter_goal_per_rc"].dropna().values
    for r in sorted(df_va_cg_rc["consent_first_count"].unique())
]

# Filter out empty groups
groups_va_cg_rc = [g for g in groups_va_cg_rc if len(g) > 0]

if len(groups_va_cg_rc) > 0:
    F_va_cg_rc, p_va_cg_rc = stats.f_oneway(*groups_va_cg_rc)
    
    k_va = len(groups_va_cg_rc)
    ns_va = [len(g) for g in groups_va_cg_rc]
    N_va = sum(ns_va)
    df_between_va = k_va - 1
    df_within_va = N_va - k_va
    eta_sq_va_cg_rc = (F_va_cg_rc * df_between_va) / (F_va_cg_rc * df_between_va + df_within_va) if (F_va_cg_rc * df_between_va + df_within_va) > 0 else 0
    
    max_cg_rc_va = df_va_cg_rc.groupby("consent_first_count")["counter_goal_per_rc"].mean().max()
    min_cg_rc_va = df_va_cg_rc.groupby("consent_first_count")["counter_goal_per_rc"].mean().min()
    
    print("Counter Goals/Resource Conflicts — MonitoringAgent (VA)")
    print(f"  F: {F_va_cg_rc:.4f}, p: {p_va_cg_rc:.3e}, eta^2: {eta_sq_va_cg_rc:.4f}")
    print(f"  Max mean: {max_cg_rc_va:.4f}, Min mean: {min_cg_rc_va:.4f}")
else:
    print("Counter Goals/Resource Conflicts — MonitoringAgent (VA)")
    print("  No valid data points")


Counter Goals/Resource Conflicts — ConsentFirstAgent (DA)
  F: 16.1706, p: 1.913e-15, eta^2: 0.6179
  Max mean: 0.1487, Min mean: 0.0914

Counter Goals/Resource Conflicts — MonitoringAgent (VA)
  F: 44.6830, p: 1.957e-29, eta^2: 0.8171
  Max mean: 0.3415, Min mean: 0.1723


/var/folders/qz/c3nhpt0n4_7g6vm6v12rwpxm0000gn/T/ipykernel_53877/361293644.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_da_cg_rc["consent_first_count"] = df_da_cg_rc["agent_config"].str.split("-").str[0].astype(int)
/var/folders/qz/c3nhpt0n4_7g6vm6v12rwpxm0000gn/T/ipykernel_53877/361293644.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_va_cg_rc["consent_first_count"] = df_va_cg_rc["agent_config"].str.split("-").str[0].astype(int)


## 12: Counter Goal Accomplishments per Resource Conflict — Mann-Whitney U Test


In [26]:
# 12: Counter Goal Accomplishments per Resource Conflict — Mann-Whitney U Test

from scipy.stats import mannwhitneyu
import pandas as pd
import numpy as np

# Build run-level averages for counter goal per resource conflict ratio
# Aggregate per run: sum counter goals and resource conflicts per persona per run, then divide
_df_cg_rc = final_agent_values.copy()

# Calculate ratio per run: sum first, then divide (matching graph calculation)
# Group by agent_config (not consent_first_count) to match the graph
run_level_cg_rc = (
    _df_cg_rc.groupby(["agent_config", "seed", "Agent Persona"])
    .agg({
        "Counter Conflict Goal Accomplishments": "sum",
        "Resource Conflicts": "sum"
    })
    .reset_index()
)

# Calculate ratio at run level (sum first, then divide) - matching graph
run_level_cg_rc["counter_goal_per_rc"] = (
    run_level_cg_rc["Counter Conflict Goal Accomplishments"] / 
    run_level_cg_rc["Resource Conflicts"].replace(0, np.nan)
)

# CRITICAL: Group by agent_config (not consent_first_count) to match the graph
# The graph groups by agent_config and averages across seeds
pivot_cg_rc = run_level_cg_rc.pivot_table(
    index=["agent_config", "seed"],
    columns="Agent Persona",
    values="counter_goal_per_rc",
    aggfunc="first"
).reset_index()

# Filter to only runs where both personas have valid values
pivot_cg_rc = pivot_cg_rc.dropna(subset=["ConsentFirstAgent", "MonitoringAgent"])

# Extract consent_first_count from agent_config for comparison table
# agent_config format: "ConsentFirst-FiftyFifty-Monitoring" (e.g., "100-0-900")
# So consent_first_count is the first part (index 0), monitoring_count is the last part (index -1)
if len(pivot_cg_rc) > 0:
    sample_config = pivot_cg_rc['agent_config'].iloc[0]
    parts = str(sample_config).split("-")
    if len(parts) >= 3:
        pivot_cg_rc["consent_first_count"] = pivot_cg_rc["agent_config"].str.split("-").str[0].astype(int)
        pivot_cg_rc["monitoring_count"] = pivot_cg_rc["agent_config"].str.split("-").str[-1].astype(int)
    elif len(parts) >= 2:
        pivot_cg_rc["consent_first_count"] = pivot_cg_rc["agent_config"].str.split("-").str[0].astype(int)
        pivot_cg_rc["monitoring_count"] = pivot_cg_rc["agent_config"].str.split("-").str[-1].astype(int)
    else:
        pivot_cg_rc["consent_first_count"] = 0
        pivot_cg_rc["monitoring_count"] = 0

# Per-condition means (grouped by agent_config, matching graph) - using only runs where both exist
cg_rc_comp_by_config = pivot_cg_rc.groupby("agent_config").agg({
    "ConsentFirstAgent": "mean",
    "MonitoringAgent": "mean",
    "consent_first_count": "first",
    "monitoring_count": "first"
}).reset_index()

# Sort by consent_first_count for display
cg_rc_comp = cg_rc_comp_by_config.sort_values("consent_first_count")[["consent_first_count", "monitoring_count", "ConsentFirstAgent", "MonitoringAgent"]].copy()
cg_rc_comp.columns = ["consent_first_count", "monitoring_count", "ConsentFirstAgent_mean", "MonitoringAgent_mean"]
cg_rc_comp["Difference"] = cg_rc_comp["ConsentFirstAgent_mean"] - cg_rc_comp["MonitoringAgent_mean"]
cg_rc_comp["ConsentFirst_higher"] = cg_rc_comp["Difference"] > 0

print("\nCounter Goals/Resource Conflicts Comparison by Persona:")
print("(Only runs where both personas exist, grouped by agent_config like the graph)")
print("=" * 80)
print(cg_rc_comp.to_string(index=False))
print("=" * 80)

da_higher = cg_rc_comp["ConsentFirst_higher"].sum()
va_higher = len(cg_rc_comp) - da_higher
print(f"\nConsentFirstAgent higher in {da_higher}/{len(cg_rc_comp)} conditions")
print(f"MonitoringAgent higher in {va_higher}/{len(cg_rc_comp)} conditions")

# For Mann-Whitney: Compare all individual run values (not condition means)
cg_rc_vals_da = pivot_cg_rc["ConsentFirstAgent"].dropna()
cg_rc_vals_va = pivot_cg_rc["MonitoringAgent"].dropna()

print("\nMann–Whitney U Test (Counter Goals/Resource Conflicts):")
print(f"  Number of runs with both personas: {len(pivot_cg_rc)}")
print(f"  ConsentFirstAgent runs: {len(cg_rc_vals_da)}")
print(f"  MonitoringAgent runs: {len(cg_rc_vals_va)}")

if len(cg_rc_vals_da) > 0 and len(cg_rc_vals_va) > 0:
    u_twosided, p_twosided = mannwhitneyu(cg_rc_vals_da, cg_rc_vals_va, alternative="two-sided")
    u_da_greater, p_da_greater = mannwhitneyu(cg_rc_vals_da, cg_rc_vals_va, alternative="greater")
    u_va_greater, p_va_greater = mannwhitneyu(cg_rc_vals_va, cg_rc_vals_da, alternative="greater")
    
    print(f"  Two-sided:           p = {p_twosided:.3e}")
    print(f"  DA > VA (one-sided): p = {p_da_greater:.3e}")
    print(f"  VA > DA (one-sided): p = {p_va_greater:.3e}")
    
    mean_da = cg_rc_vals_da.mean()
    mean_va = cg_rc_vals_va.mean()
    print(f"  ConsentFirstAgent mean CG/RC: {mean_da:.4f}")
    print(f"  MonitoringAgent    mean CG/RC: {mean_va:.4f}")
    print(f"  Difference:                   {mean_da - mean_va:.4f}")
    
    n_da = len(cg_rc_vals_da)
    n_va = len(cg_rc_vals_va)
    if n_da > 0 and n_va > 0:
        r_rb_abs = 1 - (2 * u_twosided) / (n_da * n_va)
        direction = 1 if mean_da > mean_va else -1
        r_rb = direction * abs(r_rb_abs)
        print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")
    
    if p_twosided < 0.05:
        if mean_da > mean_va:
            print("  → SIGNIFICANT: ConsentFirstAgent has higher CG/RC.")
        else:
            print("  → SIGNIFICANT: MonitoringAgent has higher CG/RC.")
    else:
        print("  → NO significant difference in CG/RC.")
else:
    print("  Insufficient data for Mann-Whitney test.")



Counter Goals/Resource Conflicts Comparison by Persona:
(Only runs where both personas exist, grouped by agent_config like the graph)
 consent_first_count  monitoring_count  ConsentFirstAgent_mean  MonitoringAgent_mean  Difference  ConsentFirst_higher
                 100               900                0.148657              0.317925   -0.169268                False
                 200               800                0.145370              0.291921   -0.146551                False
                 300               700                0.142982              0.293409   -0.150428                False
                 400               600                0.143305              0.268433   -0.125127                False
                 500               500                0.140380              0.253477   -0.113096                False
                 600               400                0.133653              0.237683   -0.104030                False
                 700               300 